# Two-Stage LoRA + Full External SmolLM3

This ensemble combines the two-stage Qwen and complementary Gaussian approach with independently trained OLMo2 and SmolLM3 classifiers. The new SmolLM3 external stage covers **28,390 distinct responses across 2,217 prompts**, followed by a separate adaptation on the official training set.

The two-stage/Gaussian branch retains its 85/15 probability mixture. OLMo2 and the new SmolLM3 contribute fixed 0.03 log-probability terms, followed by a single final balancing step. SmolLM3 uses its plain head for singletons and its centered head for recovered groups of 2–18; its optional Gaussian is excluded because it slightly reduced development accuracy.

The larger Smol warmup improved mean native-head Balanced Accuracy by **0.05877** over the earlier 5,400-response warmup on the same 539 held-out responses and three query contexts; the paired 30-prompt interval was [0.03593, 0.08210]. This is a reused development comparison, conditional on external prompt independence, not full OOF of this ensemble or a claim of a new leaderboard score.

GPU T4×2, Internet off. The pretrained language models are frozen during this notebook. The retained upstream branch fits its SVC and Gaussian readouts; the two added neural classifiers load pre-trained inference assets. The output is `submission.parquet`.

## Credits

- [Nishant Dahal](https://www.kaggle.com/code/nishantdahal/two-stage-lora-on-the-public-pool): public-pool two-stage LoRA recipe, checkpoints and external-data compilation.
- [Kirill Tymchishin](https://www.kaggle.com/code/kirilltym/0-600-teach-the-model-remove-the-prompt): the retained 85/15 two-stage and four-Gaussian combination and grouped new-Qwen Gaussian integration.
- [prvsiyan](https://www.kaggle.com/code/prvsiyan/the-prompt-is-not-the-author): shared-prompt Gaussian inference.
- Lucas Boesen, Travis Nguyen, Zbigniew Kirchner, Obstacle Developer, pjmathematician and the model/data contributors credited in the upstream notebooks and artifact notices.
- Qwen team, Allen Institute for AI and HuggingFaceTB: base models. External data retains its source attribution and CC BY-SA terms; model/source licenses remain in the attached artifacts.

## Official-only Qwen ablation

This version excludes only the older owned `lora24_public` adapter from the complementary branch: both its 0.177 plain contribution and 0.15 centered contribution. The two-stage Qwen8B/14B adapters trained on external data remain. All other coefficients and final balancing are unchanged. On the older conditional full-component validation bank, this fixed removal increased mean Balanced Accuracy from 0.54625 to 0.55745; the paired prompt interval for the difference was [-0.00095, 0.02284]. This is a diagnostic on the older ensemble, not full validation of the current 0.600 mixture.


## Load the inference pipeline

In [ ]:
import ast,hashlib,json,os,sys,subprocess
from pathlib import Path
assert os.name!='nt' and Path('/kaggle/input').is_dir()
p=Path('/kaggle/working/ensemble-source');p.mkdir()
FILES={'baseline600_runner.py': '"""Execute the exact attributed .600 model/readout prefix; save pre-Sinkhorn state."""\nimport os,json\nfrom pathlib import Path\nimport numpy as np\nfrom numeric_transport import save_numeric\nHERE=Path(__file__).resolve().parent\nsource=(HERE/\'baseline600.py\').read_text()\nconfig=json.loads((HERE/\'candidate-config.json\').read_text())\nif config.get(\'exclude_owned_official_qwen\',False):\n    from ablate_legacy_qwen import remove_owned_lora\n    source=remove_owned_lora(source)\nboundary=\'\\nP = softmax(LOGIT)\\nif len(test) >= 50:\'\nassert source.count(boundary)==1\nprefix=source.split(boundary)[0]\nnamespace={\'__name__\':\'__main__\'}\nexec(compile(prefix,\'<attributed-kirill-v20-pre-Sinkhorn>\',\'exec\'),namespace)\ncache=Path(os.environ[\'NTL_CANDIDATE_TEMP\']).resolve();assert cache.is_relative_to(Path(\'/kaggle/temp\').resolve())\ntest=namespace[\'test\'];ids=test.id.to_numpy(dtype=str)\nsave_numeric(cache/\'baseline-state.npz\',ids=ids,labels=np.asarray(namespace[\'LABELS\'],dtype=str),raw=namespace[\'LOGIT\'])\nsave_numeric(cache/\'baseline-groups.npz\',ids=ids,groups=namespace[\'groups\'],eligible=namespace[\'twin_rows\'])\nprint(\'EXACT_0600_PREFIX_AND_GROUPS_COMPLETE\',flush=True)\n', 'volume_smol_worker.py': '"""Frozen external-volume Smol inference; no optimization or Gaussian fitting."""\nimport os\nfrom pathlib import Path\nassert os.name!=\'nt\' and Path(\'/kaggle/input\').is_dir()\nimport gc,hashlib,importlib.metadata,importlib.util,json,time\nimport numpy as np\nimport pandas as pd\nfrom scipy.special import softmax\nfrom smol_readout import smol_score\nfrom numeric_transport import load_numeric,save_numeric\nfrom smol_owned_path import tokenize_owned,make_representation\nHERE=Path(__file__).resolve().parent\nS=json.loads((HERE/\'volume-spec.json\').read_text())\ndef sha(p):\n    with Path(p).open(\'rb\') as f:return hashlib.file_digest(f,\'sha256\').hexdigest()\nif os.environ.get(\'NTL_DIAGNOSTIC_VOLUME_ASSETS\'):\n    assert os.environ.get(\'NTL_RUN_MODE\')==\'diagnostic\'\n    assets=[Path(os.environ[\'NTL_DIAGNOSTIC_VOLUME_ASSETS\'])/\'volume-inference-manifest.json\']\n    assert sha(assets[0])==S[\'manifest_sha256\']\nelse:\n    assets=[p for p in Path(\'/kaggle/input\').rglob(\'volume-inference-manifest.json\') if sha(p)==S[\'manifest_sha256\']]\nassert len(assets)==1\nasset=assets[0].parent;manifest=json.loads(assets[0].read_text())\nfor name,record in manifest[\'files\'].items():assert sha(asset/name)==record[\'sha256\'],name\nassert manifest[\'public_unique_rows\']==28390 and manifest[\'public_updates\']==1696\nassert {n:importlib.metadata.version(n) for n in S[\'core_versions\']}==S[\'core_versions\']\nfinder=importlib.util.find_spec\nimportlib.util.find_spec=lambda n,*a,**kw:None if n.split(\'.\')[0]==\'torchao\' else finder(n,*a,**kw)\nimport torch\nfrom torch import nn\nfrom transformers import AutoModel\nfrom peft import PeftModel\nimport peft.import_utils\npeft.import_utils.is_torchao_available=lambda *a,**kw:False\nfrom safetensors.torch import load_file\nassert torch.cuda.device_count()==1 and torch.cuda.get_device_capability(0)==(7,5)\nbase_config=next(x for x in S[\'base_files\'] if x[\'name\']==\'config.json\')\nbases=[p.parent for p in Path(\'/kaggle/input\').rglob(\'config.json\') if sha(p)==base_config[\'sha256\']]\nassert len(bases)==1\nbase_dir=bases[0]\nfor item in S[\'base_files\']:assert sha(base_dir/item[\'name\'])==item[\'sha256\']\ncache=Path(os.environ[\'NTL_CANDIDATE_TEMP\']).resolve();assert cache.is_relative_to(Path(\'/kaggle/temp\').resolve())\ngroups=load_numeric(cache/\'baseline-groups.npz\')\ntests=[p for p in [Path(\'/kaggle/input/competitions/name-that-llm/test.csv\'),Path(\'/kaggle/input/name-that-llm/test.csv\')] if p.is_file()]\nassert len(tests)==1\nq=pd.read_csv(tests[0],dtype=str);q[\'text\']=q[\'text\'].fillna(\'\')\nnp.testing.assert_array_equal(q.id.to_numpy(dtype=str),groups[\'ids\'])\nstart=time.monotonic()\nbase,info=AutoModel.from_pretrained(base_dir,dtype=torch.float16,device_map={\'\':0},attn_implementation=\'sdpa\',local_files_only=True,output_loading_info=True)\nassert not info.get(\'missing_keys\') and not info.get(\'mismatched_keys\') and not info.get(\'error_msgs\')\nassert set(info.get(\'unexpected_keys\',[]))<={\'lm_head.weight\'}\nmodel=PeftModel.from_pretrained(base,asset/\'adapter\',is_trainable=False,local_files_only=True).eval()\nassert not any(p.requires_grad for p in model.parameters())\nnorms=nn.ModuleList([nn.LayerNorm(2048) for _ in range(6)]).cuda().float()\nsaved=load_file(asset/\'heads.safetensors\');norms.load_state_dict({k.removeprefix(\'norms.\'):v for k,v in saved.items() if k.startswith(\'norms.\')},strict=True);norms.eval()\nro=load_numeric(asset/\'readouts.npz\')\nfor route,prefix in [(\'plain\',\'plain\'),(\'center\',\'centered\')]:\n    np.testing.assert_array_equal(ro[route+\'_coef\'],saved[prefix+\'.weight\'].numpy())\n    np.testing.assert_array_equal(ro[route+\'_intercept\'],saved[prefix+\'.bias\'].numpy())\ndef features(texts,sort=True):\n    tok,tokens=tokenize_owned(base_dir,texts);fn=make_representation(model,tok,tokens,[6,12,18,24,30,36],norms)\n    order=np.argsort([len(t) for t in tokens],kind=\'stable\') if sort else np.arange(len(tokens))\n    result=np.empty((len(tokens),12288),np.float32)\n    with torch.inference_mode():\n        for i in range(0,len(order),3):\n            assert time.monotonic()-start<9000\n            ii=order[i:i+3];result[ii]=fn(ii)[0].cpu().numpy()\n    assert np.isfinite(result).all()\n    return result\nanchors=load_numeric(asset/\'synthetic-anchors.npz\');synthetic=json.loads((asset/\'synthetic3.json\').read_text())\nobserved=features(synthetic[\'texts\'],sort=False)\nnp.testing.assert_allclose(observed,anchors[\'features\'],rtol=1e-5,atol=1e-5)\nparity={}\nfor route in [\'plain\',\'center\']:\n    a=observed if route==\'plain\' else observed-observed.mean(0)\n    b=anchors[\'features\'] if route==\'plain\' else anchors[\'features\']-anchors[\'features\'].mean(0)\n    delta=float(np.abs(softmax(a@ro[route+\'_coef\'].T+ro[route+\'_intercept\'],axis=1)-softmax(b@ro[route+\'_coef\'].T+ro[route+\'_intercept\'],axis=1)).max())\n    assert delta<=1e-6;parity[route]=delta\nlabels=ro[\'classes\']\nfrom worker_barrier import ready_and_wait\nready_and_wait(\'smol\')\nbegin,end=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True)\nbegin.record();z=features(q.text.tolist());end.record();torch.cuda.synchronize()\nlg=z@ro[\'plain_coef\'].T+ro[\'plain_intercept\'];lg-=lg.max(1,keepdims=True)\np=np.exp(lg);routed=np.log(np.clip(p/p.sum(1,keepdims=True),1e-9,1.))\nlocal=np.flatnonzero(groups[\'eligible\'])\nif len(local):\n    centered=z[local].copy();local_groups=groups[\'groups\'][local]\n    for group in np.unique(local_groups):\n        mask=local_groups==group;centered[mask]-=centered[mask].mean(0)\n    value=centered@ro[\'center_coef\'].T+ro[\'center_intercept\'];value-=value.max(1,keepdims=True)\n    routed[local]=value-np.log(np.exp(value).sum(1,keepdims=True))\nscore=routed.astype(np.float64)\nsave_numeric(cache/\'smol_score.npz\',ids=groups[\'ids\'],labels=labels,score=score)\nPath(\'/kaggle/working/smol_inference_audit.json\').write_text(json.dumps({\'status\':\'FROZEN_FULL_EXTERNAL_SMOL_COMPLETE\',\'rows\':len(q),\'public_unique_rows\':28390,\'public_updates\':1696,\'model_fits\':0,\'Gaussian_fits\':0,\'synthetic_feature_maxabs\':float(np.abs(observed-anchors[\'features\']).max()),\'probability_parity\':parity,\'query_cuda_event_ms\':float(begin.elapsed_time(end)),\'seconds\':time.monotonic()-start,\'peak_reserved_bytes\':torch.cuda.max_memory_reserved(),\'CUDA_VISIBLE_DEVICES\':os.environ[\'CUDA_VISIBLE_DEVICES\']},indent=2))\n', 'run600.py': '"""Bounded two-device orchestration, exact baseline then independent owned encoders."""\nimport hashlib,json,os,signal,subprocess,sys,tempfile,time\nfrom pathlib import Path\nHERE=Path(__file__).resolve().parent\nC=json.loads((HERE/\'candidate-config.json\').read_text())\nassert os.name!=\'nt\' and Path(\'/kaggle/input\').is_dir()\nfor name,digest in C[\'source_sha256\'].items():assert hashlib.sha256((HERE/name).read_bytes()).hexdigest()==digest,name\nmode=os.environ.get(\'NTL_RUN_MODE\',\'candidate\');assert mode in (\'candidate\',\'diagnostic\')\nassert not os.environ.get(\'NTL_SMOKE\') and not os.environ.get(\'NTL_QUERY_CSV\'),\'No inherited alternate test input or pseudo-feature mode\'\nmapping=[]\nfor line in subprocess.check_output([\'nvidia-smi\',\'--query-gpu=index,uuid,name\',\'--format=csv,noheader,nounits\'],text=True).splitlines():\n    i,uuid,name=[s.strip() for s in line.split(\',\',2)];mapping.append({\'index\':int(i),\'uuid\':uuid,\'name\':name})\nmapping.sort(key=lambda x:x[\'index\']);assert [r[\'index\'] for r in mapping]==[0,1] and all(\'T4\' in r[\'name\'] for r in mapping)\nPath(\'/kaggle/temp\').mkdir(exist_ok=True);cache=Path(tempfile.mkdtemp(prefix=\'full-external-600-\',dir=\'/kaggle/temp\'))\nenv=os.environ.copy();env.update(NTL_CANDIDATE_TEMP=str(cache),HF_HUB_OFFLINE=\'1\',TRANSFORMERS_OFFLINE=\'1\',TOKENIZERS_PARALLELISM=\'false\',OMP_NUM_THREADS=\'2\',MKL_NUM_THREADS=\'2\',OPENBLAS_NUM_THREADS=\'2\')\nif mode==\'diagnostic\':env[\'NTL_DIAGNOSTIC_VOLUME_ASSETS\']=str(cache/\'volume-assets\')\ndeadline=time.monotonic()+10800\ndef remain():\n    left=deadline-time.monotonic();assert left>0,\'Three-hour bounded runtime; no retries\';return left\ndef kill(children):\n    for p in children:\n        if p.poll() is None:os.killpg(p.pid,signal.SIGTERM)\n    for p in children:\n        try:p.wait(timeout=5)\n        except subprocess.TimeoutExpired:os.killpg(p.pid,signal.SIGKILL);p.wait()\nif mode==\'candidate\':\n    script=\'baseline600_runner.py\'\nelse:\n    script=\'diagnostic600_runner.py\'\nwith Path(\'/kaggle/working/baseline.log\').open(\'w\') as log:\n    proc=subprocess.Popen([sys.executable,\'-u\',str(HERE/script)],env=env,cwd=\'/kaggle/working\',stdout=log,stderr=subprocess.STDOUT,start_new_session=True)\n    try:\n        while proc.poll() is None:remain();time.sleep(1)\n        assert proc.returncode==0,\'Baseline or mount preflight failed; see baseline.log\'\n    finally:kill([proc])\nchildren=[];handles=[];overlap=0;seen=set();start=time.monotonic()\ntry:\n    for i,(name,script) in enumerate([(\'olmo\',\'compute_olmo.py\'),(\'smol\',\'volume_smol_worker.py\')]):\n        e=env.copy();e[\'CUDA_VISIBLE_DEVICES\']=mapping[i][\'uuid\']\n        handle=Path(\'/kaggle/working\',name+\'.log\').open(\'w\');handles.append(handle)\n        children.append(subprocess.Popen([sys.executable,\'-u\',str(HERE/script)],env=e,cwd=\'/kaggle/working\',stdout=handle,stderr=subprocess.STDOUT,start_new_session=True))\n    while True:\n        remain();codes=[p.poll() for p in children];assert all(c in (None,0) for c in codes),(\'Owned inference failed\',codes)\n        if all((cache/(n+\'-READY.json\')).exists() for n in [\'olmo\',\'smol\']) and not (cache/\'OWNED-GO\').exists():\n            for i,n in enumerate([\'olmo\',\'smol\']):\n                r=json.loads((cache/(n+\'-READY.json\')).read_text());assert r[\'PID\']==children[i].pid and r[\'UUID\']==mapping[i][\'uuid\']\n            (cache/\'OWNED-GO\').touch()\n        raw=subprocess.check_output([\'nvidia-smi\',\'--query-gpu=index,uuid,utilization.gpu,memory.used\',\'--format=csv,noheader,nounits\'],text=True,timeout=10)\n        apps=subprocess.check_output([\'nvidia-smi\',\'--query-compute-apps=pid,gpu_uuid\',\'--format=csv,noheader,nounits\'],text=True,timeout=10)\n        pairs={tuple(v.strip() for v in line.split(\',\')[:2]) for line in apps.splitlines() if line.strip()}\n        matched=[(str(p.pid),mapping[i][\'uuid\']) in pairs for i,p in enumerate(children)]\n        seen.update(i for i,b in enumerate(matched) if b)\n        overlap+=int(all(matched) and (cache/\'OWNED-GO\').exists())\n        with Path(\'/kaggle/working/two-gpu-inference.jsonl\').open(\'a\') as f:f.write(json.dumps({\'elapsed\':time.monotonic()-start,\'physical_devices\':raw,\'processes\':apps,\'matched\':matched})+\'\\n\')\n        if all(c==0 for c in codes):break\n        time.sleep(.5)\n    assert seen=={0,1},\'Both GPU workers must be observed\'\nfinally:\n    kill(children)\n    for h in handles:h.close()\nsubprocess.run([sys.executable,\'-u\',str(HERE/\'finish600.py\')],env=env,cwd=\'/kaggle/working\',check=True,timeout=min(120,remain()))\nPath(\'/kaggle/working/execution-audit.json\').write_text(json.dumps({\'status\':\'COMPLETE\',\'mode\':mode,\'baseline_original_0600\':mode==\'candidate\',\'both_GPU_workers_observed\':sorted(seen),\'simultaneous_process_observations\':overlap,\'GPU_mapping\':mapping,\'retries\':0},indent=2))\n', 'finish600.py': '"""Add two independent native model terms before the single original balancing step."""\nimport os,json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom numeric_transport import load_numeric\nHERE=Path(__file__).resolve().parent;cache=Path(os.environ[\'NTL_CANDIDATE_TEMP\']).resolve()\nC=json.loads((HERE/\'candidate-config.json\').read_text())\nstate=load_numeric(cache/\'baseline-state.npz\');LOGIT=state[\'raw\'].copy();labels=state[\'labels\'].tolist();ids=state[\'ids\']\nfor name,key,weight in [(\'olmo\',\'plain_logp\',C[\'OLMo_plain_raw_coefficient\']),(\'smol\',\'score\',C[\'Smol_raw_coefficient\'])]:\n    value=load_numeric(cache/(\'olmo_plain.npz\' if name==\'olmo\' else \'smol_score.npz\'))\n    np.testing.assert_array_equal(value[\'ids\'],ids)\n    assert set(value[\'labels\'])==set(labels)\n    order=[value[\'labels\'].tolist().index(label) for label in labels]\n    term=value[key][:,order]\n    if name==\'olmo\':term=np.log(np.clip(np.exp(term),1e-9,1.))\n    LOGIT+=weight*term\nassert np.isfinite(LOGIT).all()\nsource=(HERE/\'baseline600.py\').read_text();boundary=\'\\nP = softmax(LOGIT)\\nif len(test) >= 50:\'\ntail=\'P = softmax(LOGIT)\\nif len(test) >= 50:\'+source.split(boundary)[1]\ndef softmax(x):\n    e=np.exp(x-x.max(1,keepdims=True));return e/e.sum(1,keepdims=True)\nimport time\nenv=dict(np=np,pd=pd,time=time,T0=time.time(),LOGIT=LOGIT,LABELS=labels,K=18,test=pd.DataFrame({\'id\':ids}),softmax=softmax)\nexec(compile(tail,\'<original-single-Sinkhorn100>\',\'exec\'),env)\nsub=pd.read_parquet(\'/kaggle/working/submission.parquet\');np.testing.assert_array_equal(sub.id.to_numpy(dtype=str),ids)\nassert sub.columns.tolist()==[\'id\',\'model\'] and sub.model.isin(labels).all() and len(sub)==len(ids)\nPath(\'/kaggle/working/candidate_audit.json\').write_text(json.dumps({\'status\':\'COMPLETE_0600_OLMO_FULL_EXTERNAL_SMOL\',\'rows\':len(ids),\'baseline_prefix_source_preserved\':True,\'baseline_original_prefix_executed\':os.environ.get(\'NTL_RUN_MODE\',\'candidate\')==\'candidate\',\'excluded_owned_official_qwen\':C.get(\'exclude_owned_official_qwen\',False),\'OLMo_weight\':C[\'OLMo_plain_raw_coefficient\'],\'Smol_weight\':C[\'Smol_raw_coefficient\'],\'external_unique_rows\':28390,\'full_0600_OOF_available\':False,\'single_original_Sinkhorn100\':True,\'leaderboard_score\':None},indent=2))\n', 'ablate_legacy_qwen.py': '"""Remove only the owned official-only LoRA from the old .15 probability branch."""\nimport ast\ndef remove_owned_lora(source):\n    tree=ast.parse(source)\n    node=next(n for n in tree.body if isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id==\'_GAUSSIAN_SOURCE\' for t in n.targets))\n    old=ast.literal_eval(node.value)\n    marker=\'LORA_MAX_LEN, LORA_BS = 447, 8\'\n    assert old.count(marker)==1\n    patch="\\n# Ablation: exclude only the official-only owned Qwen LoRA and its centered head.\\nW[\'lora24_public\'] = 0.0\\nLORA_TWIN_W.pop(\'lora24_public\')\\nLORA_RUNS = [(trunk, layers, [r for r in runs if r[0] != \'lora24_public\']) for trunk, layers, runs in LORA_RUNS]\\n"\n    revised=old.replace(marker,marker+patch)\n    statement=ast.get_source_segment(source,node);assert source.count(statement)==1\n    result=source.replace(statement,\'_GAUSSIAN_SOURCE = \'+repr(revised),1)\n    ast.parse(result)\n    return result\n', 'baseline600.py': 'import gc, glob, json, os, re, time\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\n\nT0 = time.time()\nINPUT_DIR = "/kaggle/input"\n\n# Blend weights, fit on out-of-fold predictions (log probabilities are multiplied by these and summed).\nW_SVC = 1.021\nLORAS = {  # checkpoint folder: blend weight, centred-twin weight, batch size, base model to look for\n    "lora8b_v1":  dict(w=0.226, twin_w=0.079, bs=16, base=dict(model_type="qwen3", hidden_size=4096, num_hidden_layers=36)),\n    "lora14b_v1": dict(w=0.429, twin_w=0.150, bs=8,  base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),\n}\nTWIN = dict(thresh=0.85, min_size=2, max_size=18)   # topic clusters that get the centred twin\n\n# Column order of the checkpoints\' heads (checked against each checkpoint\'s meta.json below).\nLABELS = [\'gpt-5.5\', \'gpt-5.6-luna\', \'gpt-5.6-sol\', \'gpt-5.6-terra\', \'gpt-6-astra\',\n          \'gemini-3.1-pro-preview\', \'gemini-3.6-flash\', \'gemini-3.7-flash\', \'gemini-3.8-flash\',\n          \'claude-opus-4-6\', \'claude-opus-4-8\', \'claude-opus-5\', \'claude-sonnet-5\', \'claude-fable-5\',\n          \'qwen3.8-max\', \'glm-5.3\', \'glm-5.3-flash\', \'kimi-k3\']\nK = len(LABELS)\n\n\ndef find(pattern):\n    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))\n    if not hits:\n        raise FileNotFoundError(pattern)\n    return hits[0]\n\n\ndef softmax(x):\n    e = np.exp(x - x.max(1, keepdims=True))\n    return e / e.sum(1, keepdims=True)\n\n\n_competition_dir = next(p for p in ["/kaggle/input/competitions/name-that-llm", "/kaggle/input/name-that-llm"] if os.path.exists(p + "/train.csv"))\ntrain = pd.read_csv(_competition_dir + "/train.csv")\ntest = pd.read_csv(_competition_dir + "/test.csv")\ntest["text"] = test["text"].fillna("").astype(str)\ny = train["model"].map({l: i for i, l in enumerate(LABELS)}).values\nLOGIT = np.zeros((len(test), K))   # every member adds w * log p here\nprint("train", train.shape, "test", test.shape, flush=True)\n\n\nfrom scipy.sparse import hstack\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.svm import LinearSVC\n\nFUNCTION_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())\nVIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("function", 1, 3, 50000, 0.5)]\n\n\ndef shape_text(t):\n    return re.sub(r"[^\\W\\d_]+", lambda m: "X" if m[0].isupper() else "x", t)\n\n\ndef function_text(t):\n    return " ".join(w.lower() if w.lower() in FUNCTION_WORDS or not w.isalnum() else "CONTENT"\n                    for w in re.findall(r"\\w+|[^\\w\\s]", t))\n\n\npool = pd.read_parquet(find("pool_v1.parquet"))\nfit_texts = train.text.astype(str).tolist() + pool.text.tolist()\nfit_y = np.r_[y, pool.model.map({l: i for i, l in enumerate(LABELS)}).values]\nA, B = [], []\nfor view, lo, hi, max_features, weight in VIEWS:\n    prep = shape_text if view == "shape" else function_text if view == "function" else (lambda s: s)\n    analyzer = "char" if view == "shape" else "word" if view == "function" else view\n    opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,\n                max_features=max_features, dtype=np.float32)\n    if analyzer == "word":\n        opts["token_pattern"] = r"(?u)\\b\\w+\\b|[^\\w\\s]"\n    vec = TfidfVectorizer(**opts)\n    A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)\n    B.append(vec.transform([prep(t) for t in test.text]) * weight)\nsvc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced").fit(hstack(A, format="csr"), fit_y)\nP_svc = softmax(svc.decision_function(hstack(B, format="csr")))\nLOGIT += W_SVC * np.log(np.clip(P_svc, 1e-9, 1))\ndel A, B, svc, pool\ngc.collect()\nprint(f"n-gram model done ({time.time() - T0:.0f}s)", flush=True)\n\n\nfrom safetensors.torch import load_file\nfrom transformers import AutoConfig, AutoModel, AutoTokenizer\n\n\ndef find_base(spec):\n    for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):\n        try:\n            c = json.load(open(cfg))\n        except Exception:\n            continue\n        if all(c.get(k) == v for k, v in spec.items()):\n            return os.path.dirname(cfg)\n    raise FileNotFoundError(f"attach the base model {spec}")\n\n\ndef load_trunk(base_dir, keep, dtype):\n    """First `keep` decoder layers, split over both T4s when it does not fit on one."""\n    mc = AutoConfig.from_pretrained(base_dir)\n    mc.num_hidden_layers, mc.use_cache = keep, False\n    if getattr(mc, "layer_types", None) is not None:\n        mc.layer_types = mc.layer_types[:keep]\n    hd = getattr(mc, "head_dim", mc.hidden_size // mc.num_attention_heads)\n    gib = 2 * (mc.vocab_size * mc.hidden_size + keep * (2 * mc.hidden_size * mc.num_attention_heads * hd\n               + 2 * mc.hidden_size * mc.num_key_value_heads * hd + 3 * mc.hidden_size * mc.intermediate_size)) / 2 ** 30\n    if gib > 13 and torch.cuda.device_count() > 1:\n        mem = {i: "13GiB" for i in range(torch.cuda.device_count())}\n        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto", max_memory=mem).eval()\n    else:\n        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to("cuda:0").eval()\n    model.norm = nn.Identity()\n    return model\n\n\n@torch.no_grad()\ndef merge_lora(model, adapter_dir):\n    acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))\n    scale = acfg["lora_alpha"] / acfg["r"]\n    sd = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))\n    modules = dict(model.named_modules())\n    for k, A in sd.items():\n        if ".lora_A." in k:\n            name = k.split(".lora_A.")[0]\n            lin = modules[name[name.index("layers."):]]\n            Bm = sd[k.replace(".lora_A.", ".lora_B.")]\n            lin.weight.add_(((Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale).to(lin.weight.dtype))\n\n\n@torch.no_grad()\ndef lora_features(ckpt_dir, base_dir, texts, bs):\n    """Pooled features Z (n, 8 * hidden) and the checkpoint\'s heads."""\n    meta = json.load(open(os.path.join(ckpt_dir, "meta.json")))\n    assert meta["labels"] == LABELS, "label order differs from the checkpoint\'s heads"\n    cfg = meta["cfg"]\n    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16\n    tok = AutoTokenizer.from_pretrained(base_dir)\n    tok.padding_side = "right"\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    model = load_trunk(base_dir, cfg["keep"], dtype)\n    merge_lora(model, os.path.join(ckpt_dir, "adapter"))\n    heads = load_file(os.path.join(ckpt_dir, "heads.safetensors"))\n    layers, H = cfg["layers"], model.config.hidden_size\n    norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()\n    norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})\n    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg["max_len"])["input_ids"]\n    ids = [x + [tok.eos_token_id] for x in ids]\n    order = np.argsort([len(x) for x in ids], kind="stable")   # similar lengths together = less padding\n    Z = np.zeros((len(ids), H * len(layers)), np.float32)\n    for s in range(0, len(order), bs):\n        idx = order[s:s + bs]\n        enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)\n        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states\n        if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow: redo this batch in bf16\n            model.to(torch.bfloat16)\n            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states\n            model.to(dtype)\n        m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")\n        m[:, 0] = 0   # the first token is an attention sink; leave it out of the mean\n        den = m.sum(1).clamp(min=1)\n        Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()\n    del model\n    torch.cuda.empty_cache()\n    return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith("norms.")}\n\n\nfrom scipy.cluster.hierarchy import fcluster, linkage\nfrom scipy.spatial.distance import squareform\n\nXt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test.text.tolist())\nD = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)\nnp.fill_diagonal(D, 0)\ngroups = fcluster(linkage(squareform(D, checks=False), method="average"), TWIN["thresh"], criterion="distance") \\\n    if len(test) >= 2 else np.ones(len(test), dtype=int)\nsize = np.bincount(groups)[groups]\ntwin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])\ndel D\nprint(f"topic clusters: {int((size == 1).sum())} singletons, {int(twin_rows.sum())} passages in clusters of 2-18", flush=True)\n\n# Gaussian shared-prompt core, credited to prvsiyan.\nimport itertools\nK = 18\nDEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,\n           "iters": 5, "repel": True, "sp_scale": 1.0}\n\n# =================================================================== primitives ====\ndef _lse(A, axis=None, keepdims=False):\n    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)\n    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m\n    if keepdims: return out\n    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())\n\ndef log_softmax(L): return L - _lse(L, axis=1, keepdims=True)\n\ndef _lw_shrinkage(X):\n    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""\n    from sklearn.covariance import ledoit_wolf_shrinkage\n    return float(ledoit_wolf_shrinkage(X, assume_centered=True))\n\ndef _shrunk(S, a):\n    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)\n\ndef _two_way(Z, y, pinv, G, iters=4):\n    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""\n    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)\n    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))\n    for _ in range(iters):\n        R = Z - P[pinv]\n        for c in cls: mu[c] = R[y == c].mean(0)\n        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]\n    return mu, P, cnt_p\n\ndef fit(X, y, prompts, cfg=None):\n    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.\n    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller\'s cross-fit)."""\n    c = dict(DEFAULT); c.update(cfg or {})\n    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape\n    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)\n    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx\n    if c["red"] == "none":\n        V = np.eye(D)\n    else:\n        if c["red"] == "wpca":\n            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]\n        elif c["red"] == "pca": R = Xs\n        else: raise ValueError(c["red"])\n        d = int(min(c["d"], N - G - 1, D))\n        # economy SVD through the (N x N) Gram matrix (N << D for the real views)\n        if N < D:\n            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]\n            V = (R.T @ U) / np.sqrt(w)\n        else:\n            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T\n    Z = Xs @ V; d = Z.shape[1]\n    mu, P, ng = _two_way(Z, y, pinv, G)\n    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)\n    Se_raw = E.T @ E / dof\n    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)\n    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw\n    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])\n    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]\n    # joint diagonalisation: A Se A\' = I, A Sp A\' = diag(lam)\n    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)\n    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li\n    W = V @ A.T\n    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,\n            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}\n\ndef transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]\n\n_MASK = {}\ndef _inj_mask(n, C):\n    if (n, C) not in _MASK:\n        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)\n        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]\n        _MASK[(n, C)] = bad\n    return _MASK[(n, C)]\n\ndef singleton_ll(U, m, lam):\n    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""\n    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)\n\ndef joint_terms(U, m, lam):\n    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""\n    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]\n    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)\n    return a, B\n\ndef joint_table(a, B):\n    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""\n    n, C = a.shape; Q = np.zeros([C] * n)\n    for i in range(n):\n        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)\n    for i, j in itertools.combinations(range(n), 2):\n        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)\n    return Q\n\ndef exact_marginals(Q, T=1.0, inj=True):\n    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""\n    n = Q.ndim; C = Q.shape[0]; Qt = Q / T\n    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)\n    lz = _lse(Qt); out = np.empty((n, C))\n    for i in range(n):\n        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz\n    return out\n\ndef em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):\n    """cavity EM for one group (any n >= 2): returns (n, C) log q."""\n    n = len(U); v = lam / (1 + (n - 1) * lam)\n    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))\n    q = np.exp(log_softmax(L))\n    for _ in range(iters):\n        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)\n        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T\n        if repel:   # mean-field at-most-once: discourage classes the other members already hold\n            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)\n        q = np.exp(log_softmax(L))\n    return log_softmax(L)\n\nLAM_DIFF = 1e6\ndef lam_g(prm, cfg=None):\n    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg \'diff\'): then the joint likelihood\n    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs\n    centred prototypes) and the prompt term cancels exactly."""\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})\n    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]\n\ndef group_logpost(U, prm, T1=None, Tg=None, cfg=None):\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)\n    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg\n    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)\n    if n <= c["n_exact"]:\n        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])\n    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])\n\ndef scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):\n    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""\n    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))\n    sizes = np.bincount(lab); s1 = sizes[lab] == 1\n    T1 = prm["T1"] if T1 is None else T1\n    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)\n    for g in np.where(sizes >= 2)[0]:\n        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)\n    return out\n\n\nfor ckpt_name, mc in LORAS.items():\n    ckpt = os.path.dirname(find(f"{ckpt_name}/meta.json"))\n    if ckpt_name == \'lora8b_v1\':\n        # One 8B trunk load for official-train and test features; classifier checkpoints remain unchanged.\n        _all_texts = test.text.tolist() + train.text.astype(str).tolist()\n        _all_Z, heads = lora_features(ckpt, find_base(mc[\'base\']), _all_texts, mc[\'bs\'])\n        Z, _train_Z = _all_Z[:len(test)], _all_Z[len(test):]\n        _prm = fit(_train_Z, y, train[\'prompt_id\'].values,\n                   dict(red=\'wpca\', d=128, se_shrink=\'lw\', sp_shrink=\'lw\', lam_floor=1e-4,\n                        n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0))\n        _prm[\'T1\'] = _prm[\'Tg\'] = 3.0\n        if twin_rows.any():\n            _lp = scores(Z[twin_rows], np.unique(groups[twin_rows], return_inverse=True)[1], _prm)\n            assert np.isfinite(_lp).all()\n            LOGIT[twin_rows] += 0.10 * _lp\n            print(\'NEW_LORA_GAUSSIAN rows\', int(twin_rows.sum()), \'d128 T3 weight.10\', flush=True)\n        else:\n            print(\'NEW_LORA_GAUSSIAN fitted; no eligible public-test groups\', flush=True)\n        # Exercise exact pair inference even on the three-row example, without touching its predictions.\n        _pid = train[\'prompt_id\'].values\n        _ii = np.flatnonzero(_pid == _pid[0])[:2]\n        assert len(_ii) == 2\n        _qa = scores(_train_Z[_ii], np.zeros(2, dtype=int), _prm)\n        assert _qa.shape == (2, K) and np.isfinite(_qa).all() and np.allclose(np.exp(_qa).sum(1), 1)\n        print(\'NEW_LORA_GAUSSIAN exact-pair probability check PASS\', flush=True)\n        del _all_Z, _train_Z, _prm, _qa, _all_texts\n    else:\n        Z, heads = lora_features(ckpt, find_base(mc[\'base\']), test.text.tolist(), mc[\'bs\'])\n    LOGIT += mc["w"] * np.log(np.clip(softmax(Z @ heads["plain.weight"].T + heads["plain.bias"]), 1e-9, 1))\n    if twin_rows.any():\n        Zc = Z.copy()\n        for g in np.unique(groups[twin_rows]):\n            Zc[groups == g] -= Zc[groups == g].mean(0)\n        centred = Zc[twin_rows] @ heads["centered.weight"].T + heads["centered.bias"]\n        LOGIT[twin_rows] += mc["twin_w"] * np.log(np.clip(softmax(centred), 1e-9, 1))\n    del Z\n    gc.collect()\n    print(f"{ckpt_name} done ({time.time() - T0:.0f}s)", flush=True)\n\n\n_GAUSSIAN_SOURCE = \'# ============================================================ 0. configuration\\nimport gc, glob, json, os, pathlib, re, time\\nimport numpy as np, pandas as pd\\ntry:\\n    import torch\\nexcept ImportError:   # local smoke without torch\\n    torch = None\\nfrom scipy.cluster.hierarchy import fcluster, linkage\\nfrom scipy.sparse import csr_matrix, diags, hstack, vstack\\nfrom scipy.spatial.distance import squareform\\nfrom sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.preprocessing import StandardScaler\\nfrom sklearn.svm import LinearSVC\\n\\nSMOKE = os.environ.get(\\\'NTL_SMOKE\\\') == \\\'1\\\'   # local code-path check: pseudo features instead of checkpoints, CPU\\nT0 = time.time()\\nLABELS = sorted([\\\'gpt-5.5\\\', \\\'gpt-5.6-luna\\\', \\\'gpt-5.6-sol\\\', \\\'gpt-5.6-terra\\\', \\\'gpt-6-astra\\\',\\n                 \\\'gemini-3.1-pro-preview\\\', \\\'gemini-3.6-flash\\\', \\\'gemini-3.7-flash\\\', \\\'gemini-3.8-flash\\\',\\n                 \\\'claude-fable-5\\\', \\\'claude-opus-4-6\\\', \\\'claude-opus-4-8\\\', \\\'claude-opus-5\\\', \\\'claude-sonnet-5\\\',\\n                 \\\'qwen3.8-max\\\', \\\'glm-5.3\\\', \\\'glm-5.3-flash\\\', \\\'kimi-k3\\\'])\\nK = len(LABELS)\\nLAB2I = {l: i for i, l in enumerate(LABELS)}\\nFAMILY_IDX = {f: np.array([i for i, l in enumerate(LABELS) if l.startswith(f)]) for f in (\\\'claude\\\', \\\'gpt\\\', \\\'gemini\\\', \\\'glm\\\')}\\n\\n# ---- test-pool grouping\\nGROUP_THRESH, GROUP_LAYER, GROUP_MIN, GROUP_MAX = 0.60, 24, 2, 18   # mixed content + GLM-L24 distance, average linkage\\nSTYLE_GROUP_THRESH = 0.90                                           # stylegram\\\'s own median-normalised content distance\\n\\n# ---- blend weights: log-probability multipliers of the 0.563 notebook (its L-BFGS fit on prompt-grouped OOF)\\nW = {\\\'stylegram\\\': 1.516, \\\'stylometry_lgb\\\': 0.063, \\\'ultrabert\\\': 0.095,\\n     \\\'q3_8b\\\': 0.0, \\\'q25_7b\\\': 0.0, \\\'q3_4b\\\': 0.0, \\\'glm9b\\\': 0.0, \\\'debv3l\\\': 0.05, \\\'electl\\\': 0.032, \\\'desk\\\': 0.132,\\n     \\\'lora24_public\\\': 0.177, \\\'lora24_seed7\\\': 0.113, \\\'lora24_claude\\\': 0.177, \\\'lora20_q25_instyle\\\': 0.17,\\n     \\\'tokattn_q35\\\': 0.153, \\\'tokattn_deb\\\': 0.185}\\nTWIN_W = {\\\'q3_8b\\\': 1.5 * 0.219, \\\'q25_7b\\\': 1.5 * 0.141, \\\'q3_4b\\\': 1.5 * 0.083, \\\'glm9b\\\': 1.5 * 0.110, \\\'debv3l\\\': 0.25}\\nLORA_TWIN_W = {\\\'lora24_public\\\': 0.15, \\\'lora24_claude\\\': 0.15}\\nGATE_LAM = {\\\'claude_style\\\': 1.5, \\\'claude_lora\\\': 1.0, \\\'family\\\': 0.25}\\nBALANCE_ALPHA = 0.5\\nEXTRA_MIN_LABELS, EXTRA_SHARE = 12, 0.5\\n\\n# ---- the addition: shared-prompt Gaussian group heads on four measured views, weight = the centred twin of the same backbone,\\n#      reduction / temperatures from our production fit (research/gaussian/production_weights_v8g.json, 135 training prompts)\\nGAUSS_HEADS = {\\\'gauss_q3_8b_L22\\\': dict(src=\\\'q3_8b\\\', part=\\\'L22_mean\\\', d=384, T1=2.351, Tg=2.306, w=TWIN_W[\\\'q3_8b\\\']),\\n               \\\'gauss_glm9b_L8\\\': dict(src=\\\'glm9b\\\', part=\\\'L8_mean\\\', d=384, T1=2.722, Tg=2.596, w=TWIN_W[\\\'glm9b\\\']),\\n               \\\'gauss_q3_4b_L4\\\': dict(src=\\\'q3_4b\\\', part=\\\'L4_mean\\\', d=384, T1=2.725, Tg=2.578, w=TWIN_W[\\\'q3_4b\\\']),\\n               \\\'gauss_debv3l_L24\\\': dict(src=\\\'debv3l\\\', part=\\\'L24_mean\\\', d=384, T1=3.112, Tg=3.069, w=TWIN_W[\\\'debv3l\\\'])}\\nGAUSS_SETTING = dict(red=\\\'wpca\\\', se_shrink=\\\'lw\\\', sp_shrink=\\\'lw\\\', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0)\\n\\n# ---- member specifications\\nLM_HEADS = [  # key, model_type, hidden size, parts of the plain head [(layer, pool)], extra parts extracted in the same pass, C\\n    (\\\'glm9b\\\', \\\'glm4\\\', 4096, [(8, \\\'mean\\\'), (3, \\\'mean\\\')], [(GROUP_LAYER, \\\'mean\\\')], 0.0025),   # first: its L24 builds the groups\\n    (\\\'q3_8b\\\', \\\'qwen3\\\', 4096, [(7, \\\'mean\\\'), (22, \\\'mean\\\')], [], 0.0025),\\n    (\\\'q25_7b\\\', \\\'qwen2\\\', 3584, [(6, \\\'mean\\\'), (20, \\\'mean\\\')], [], 0.0025),\\n    (\\\'q3_4b\\\', \\\'qwen3\\\', 2560, [(4, \\\'style\\\'), (7, \\\'style\\\')], [(4, \\\'mean\\\')], 0.0025)]\\nENC_HEADS = [  # key, input directory name, parts, C\\n    (\\\'debv3l\\\', \\\'deberta-v3-large\\\', [(24, \\\'mean\\\'), (6, \\\'mean\\\')], 0.005),\\n    (\\\'electl\\\', \\\'electra-large-discriminator\\\', [(21, \\\'mean\\\'), (24, \\\'style\\\')], 0.005),\\n    (\\\'desk\\\', \\\'desklib-detector\\\', [(23, \\\'mean\\\'), (23, \\\'style\\\')], 0.005)]\\nLORA_RUNS = [  # (trunk model_type, hidden, kept layers), pooled layers, [(key, checkpoint marker, norms file, anchor features file)]\\n    ((\\\'qwen3\\\', 4096, 24), [2, 4, 6, 8, 12, 16, 20, 24], [\\n        (\\\'lora24_public\\\', \\\'**/lora24/final/COMPLETE.json\\\', \\\'heads.safetensors\\\', \\\'anchor_features.npz\\\'),\\n        (\\\'lora24_seed7\\\', \\\'**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json\\\', \\\'norms.safetensors\\\', \\\'valid_features.npz\\\'),\\n        (\\\'lora24_claude\\\', \\\'**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json\\\', \\\'heads.safetensors\\\', \\\'anchor_features.npz\\\')]),\\n    ((\\\'qwen2\\\', 3584, 20), [2, 4, 6, 8, 10, 12, 16, 20], [\\n        (\\\'lora20_q25_instyle\\\', \\\'**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json\\\', \\\'norms.safetensors\\\', \\\'valid_features.npz\\\')])]\\nLORA_MAX_LEN, LORA_BS = 447, 8\\nTOKATTN_HEADS = [(\\\'tokattn_q35\\\', \\\'tokattn_final.pt\\\', \\\'qwen3_5\\\'), (\\\'tokattn_deb\\\', \\\'tokattn_debl23_final.pt\\\', \\\'deberta\\\')]\\n\\n# ---- data\\nif SMOKE:\\n    _all = pd.read_csv(os.environ[\\\'NTL_DATA\\\'] + \\\'/train.csv\\\')\\n    _p = _all[\\\'prompt_id\\\'].drop_duplicates().sample(frac=1, random_state=0).tolist()\\n    train = _all[_all[\\\'prompt_id\\\'].isin(_p[:30])].reset_index(drop=True)\\n    test = _all[_all[\\\'prompt_id\\\'].isin(_p[30:42])].sample(frac=1, random_state=1).reset_index(drop=True)\\n    test = pd.concat([test[test[\\\'prompt_id\\\'] == p].head(k) for p, k in zip(_p[30:42], [1, 1, 2, 2, 2, 3, 3, 4, 6, 9, 18, 18])]).reset_index(drop=True)\\n    if os.environ.get(\\\'NTL_TINY\\\') == \\\'1\\\':\\n        test = test.head(3).reset_index(drop=True)\\n    SMOKE_Y = test[\\\'model\\\'].map(LAB2I).to_numpy(dtype=int)\\nelse:\\n    DATA_DIR = next(p for p in [pathlib.Path(\\\'/kaggle/input/competitions/name-that-llm\\\'), pathlib.Path(\\\'/kaggle/input/name-that-llm\\\')]\\n                    if (p / \\\'train.csv\\\').exists())\\n    train, test = pd.read_csv(DATA_DIR / \\\'train.csv\\\'), pd.read_csv(DATA_DIR / \\\'test.csv\\\')\\ntest[\\\'text\\\'] = test[\\\'text\\\'].fillna(\\\'\\\').astype(str)\\nTR, TE = train[\\\'text\\\'].tolist(), test[\\\'text\\\'].tolist()\\ny = train[\\\'model\\\'].map(LAB2I).to_numpy(dtype=int)\\nPROMPT = train[\\\'prompt_id\\\'].to_numpy(dtype=object)\\nTINY = len(TE) < 50   # the interactive 3-row test: exercise the group paths on training rows so the hidden rerun cannot hit untested code\\nDEV = \\\'cuda\\\' if torch is not None and torch.cuda.is_available() else \\\'cpu\\\'\\nprint(f\\\'train {train.shape}  test {test.shape}  gpus {torch.cuda.device_count() if torch is not None else 0}  smoke {SMOKE}\\\', flush=True)\\n\\nLOGIT = np.zeros((len(TE), K))           # the running log-probability blend\\n\\n\\ndef add_member(key, prob, rows=None, weight=None):\\n    """Add weight * log(prob) to the blend (all rows, or the given row subset)."""\\n    w = W[key] if weight is None else weight\\n    lp = np.log(np.clip(prob, 1e-9, 1.0))\\n    if rows is None:\\n        LOGIT[:] += w * lp\\n    else:\\n        LOGIT[rows] += w * lp\\n    print(f\\\'  + {key:20s} weight {w:.3f} on {len(TE) if rows is None else int(np.sum(rows))} rows  ({time.time() - T0:.0f}s)\\\', flush=True)\\n\\n\\ndef softmax_rows(S):\\n    S = S - S.max(1, keepdims=True)\\n    P = np.exp(S)\\n    return P / P.sum(1, keepdims=True)\\n\\n\\ndef full_prob(P_part, classes):\\n    """Expand a (n, len(classes)) probability block to (n, K) on the global label order."""\\n    P = np.zeros((len(P_part), K))\\n    P[:, classes] = P_part\\n    return P\\n\\n\\ndef group_centre_dense(X, groups):\\n    X = np.asarray(X, dtype=np.float32).copy()\\n    for g in np.unique(groups):\\n        i = groups == g\\n        X[i] -= X[i].mean(0)\\n    return X\\n\\n\\ndef self_test_rows():\\n    """Training rows of the first three prompts, grouped by their true prompt (used only when the test file is tiny)."""\\n    idx = np.where(np.isin(PROMPT, pd.unique(PROMPT)[:3]))[0]\\n    return idx, np.unique(PROMPT[idx], return_inverse=True)[1]\\n\\n# ============================================================ 1. grouping the test pool\\ndef average_linkage_groups(D, thresh):\\n    """Cluster ids (1-based) from a square distance matrix, average linkage, cut at thresh."""\\n    if len(D) < 2:\\n        return np.ones(len(D), dtype=int)\\n    D = np.asarray(D, dtype=np.float64).copy()\\n    np.fill_diagonal(D, 0.0)\\n    return fcluster(linkage(squareform(D, checks=False), method=\\\'average\\\'), thresh, criterion=\\\'distance\\\')\\n\\n\\ndef content_cosine_distance(texts, max_features=None):\\n    X = TfidfVectorizer(stop_words=\\\'english\\\', sublinear_tf=True, max_features=max_features, dtype=np.float32).fit_transform(texts)\\n    return np.clip(1.0 - (X @ X.T).toarray(), 0.0, 1.0)\\n\\n\\ndef mixed_groups(texts, E, thresh=GROUP_THRESH):\\n    """Average of the content-word TF-IDF cosine distance and the cosine distance of mean-centred, L2-normalised embeddings E."""\\n    if len(texts) < 2:\\n        return np.ones(len(texts), dtype=int)\\n    Z = np.asarray(E, dtype=np.float64)\\n    Z = Z - Z.mean(0)\\n    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12\\n    D = 0.5 * (content_cosine_distance(texts) + np.clip(1.0 - Z @ Z.T, 0.0, 2.0))\\n    return average_linkage_groups(D, thresh)\\n\\n\\ndef normalised_content_groups(texts, thresh=STYLE_GROUP_THRESH):\\n    """The stylegram\\\'s grouping: content distance divided by its median off-diagonal value, cut at thresh."""\\n    if len(texts) < 2:\\n        return np.ones(len(texts), dtype=int)\\n    D = content_cosine_distance(texts, max_features=100_000)\\n    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-6)\\n    return average_linkage_groups(D / med, thresh)\\n\\n\\ndef eligible(groups, lo=GROUP_MIN, hi=GROUP_MAX):\\n    size = np.bincount(groups)[groups]\\n    return (size >= lo) & (size <= hi)\\n\\n\\ndef describe_groups(name, groups):\\n    size = np.bincount(groups)[groups]\\n    print(f\\\'{name}: {len(groups)} rows | singletons {int((size == 1).sum())}  2-3: {int(((size >= 2) & (size <= 3)).sum())}\\\'\\n          f\\\'  4-18: {int(((size >= 4) & (size <= 18)).sum())}  >18: {int((size > 18).sum())}\\\', flush=True)\\n\\n\\nG_TE, GROUP_ROWS = None, None   # set after the GLM pass (section 4)\\n\\n# ============================================================ 2. style n-grams + extra passages + gates\\nFUNCTION_WORDS = frozenset("a an the and or but if while although because as so than that which who whom whose this these those it its "\\n                           "they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with "\\n                           "without for of by over under between through into about is are was were be been being have has had do does did "\\n                           "can could will would shall should may might must not no nor all any some each every both either neither other "\\n                           "another such only also just very too more most less least here there when where why how what whether then now "\\n                           "still yet even much many few little".split())\\nTOKEN_RE = re.compile(r"\\\\w+|[^\\\\w\\\\s]")\\n\\n\\ndef shape_view(text):\\n    return re.sub(r"[^\\\\W\\\\d_]+", lambda m: \\\'X\\\' if m[0].isupper() else \\\'x\\\', text)\\n\\n\\ndef function_view(text):\\n    return \\\' \\\'.join(t.lower() if (t.lower() in FUNCTION_WORDS or not t.isalnum()) else \\\'CONTENT\\\' for t in TOKEN_RE.findall(text))\\n\\n\\nclass StyleGrams:\\n    """Four stacked TF-IDF views: char 3-5, word 1-3, shape (char 2-6 of the x/X text), function-word (word 1-3 of the masked text)."""\\n    VIEWS = [(\\\'char\\\', str, (3, 5), 250_000, 1.0), (\\\'word\\\', str, (1, 3), 180_000, 0.7),\\n             (\\\'char\\\', shape_view, (2, 6), 60_000, 0.5), (\\\'word\\\', function_view, (1, 3), 50_000, 0.5)]\\n\\n    def fit(self, texts):\\n        self.vecs = []\\n        for analyzer, view, ngr, maxf, w in self.VIEWS:\\n            opts = dict(analyzer=analyzer, ngram_range=ngr, lowercase=False, sublinear_tf=True, min_df=2, max_features=maxf, dtype=np.float64)\\n            if analyzer == \\\'word\\\':\\n                opts[\\\'token_pattern\\\'] = r"(?u)\\\\b\\\\w+\\\\b|[^\\\\w\\\\s]"\\n            self.vecs.append((TfidfVectorizer(**opts).fit([view(t) for t in texts]), view, w))\\n        return self\\n\\n    def transform(self, texts):\\n        return hstack([v.transform([view(t) for t in texts]) * w for v, view, w in self.vecs], format=\\\'csr\\\')\\n\\n\\ndef sparse_group_centre(X, groups, lo=GROUP_MIN, hi=GROUP_MAX):\\n    """Subtract each group\\\'s mean row (sparse); groups outside [lo, hi] are left as they are."""\\n    X = csr_matrix(X)\\n    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)\\n    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))\\n    if not len(keep):\\n        return X.copy()\\n    col = np.full(len(cnt), -1); col[keep] = np.arange(len(keep))\\n    rows = np.flatnonzero(col[inv] >= 0)\\n    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))\\n    means = diags(1.0 / cnt[keep]) @ (M.T @ X)\\n    return (X - M @ means).tocsr()\\n\\n\\ndef svc_probs(svc, X):\\n    return full_prob(softmax_rows(svc.decision_function(X)), svc.classes_)\\n\\n\\n# ---- public in-style passages (prompt-bank protocol), one per (prompt, label); prompts answered by >= EXTRA_MIN_LABELS labels\\ndef load_extra_passages():\\n    sources = [(\\\'**/ntl_extra_passages_claude.csv\\\', \\\'model\\\', None), (\\\'**/gen_passages.csv\\\', \\\'model\\\', None),\\n               (\\\'**/ntl_generations.csv\\\', \\\'label\\\', \\\'finish_reason\\\'), (\\\'**/ntl_proxy_generations.csv\\\', \\\'label\\\', None),\\n               (\\\'**/ntl_generated*.csv\\\', \\\'model\\\', \\\'finish\\\'), (\\\'**/passages_clean.csv\\\', \\\'model\\\', None)]\\n    parts = []\\n    for pattern, label_col, finish_col in sources:\\n        for f in sorted(glob.glob(\\\'/kaggle/input/\\\' + pattern, recursive=True)):\\n            d = pd.read_csv(f)\\n            if finish_col and finish_col in d:\\n                d = d[d[finish_col].eq(\\\'stop\\\')]\\n            d = d.rename(columns={label_col: \\\'model\\\'})[[\\\'prompt_id\\\', \\\'model\\\', \\\'text\\\']].copy()\\n            d[\\\'prompt_id\\\'] = (\\\'pv_\\\' if f.endswith(\\\'passages_clean.csv\\\') else \\\'\\\') + d[\\\'prompt_id\\\'].astype(str)   # prvsiyan\\\'s prompts are a different bank\\n            parts.append(d)\\n    if not parts:\\n        return pd.DataFrame(columns=[\\\'prompt_id\\\', \\\'model\\\', \\\'text\\\'])\\n    x = pd.concat(parts, ignore_index=True)\\n    x = x[x[\\\'model\\\'].isin(LABELS)].copy()\\n    x[\\\'text\\\'] = x[\\\'text\\\'].astype(str).map(lambda t: re.sub(r\\\'\\\\s+\\\', \\\' \\\', re.sub(r\\\'^\\\\*\\\\*(.*)\\\\*\\\\*$\\\', r\\\'\\\\1\\\', t.strip())).strip())\\n    x = x[x[\\\'text\\\'].str.split().str.len().between(30, 300)].drop_duplicates(subset=[\\\'model\\\', \\\'text\\\'])\\n    x = x.drop_duplicates(subset=[\\\'prompt_id\\\', \\\'model\\\'])\\n    x = x[x.groupby(\\\'prompt_id\\\')[\\\'model\\\'].transform(\\\'nunique\\\') >= EXTRA_MIN_LABELS].reset_index(drop=True)\\n    return x\\n\\n\\nEXTRA = load_extra_passages()\\nEX_T, EX_Y, EX_G = EXTRA[\\\'text\\\'].tolist(), EXTRA[\\\'model\\\'].map(LAB2I).to_numpy(dtype=int), EXTRA[\\\'prompt_id\\\'].to_numpy(dtype=object)\\nprint(f\\\'extra passages: {len(EXTRA)} rows, {EXTRA["prompt_id"].nunique()} prompts\\\', flush=True)\\n\\n\\ndef extra_sample_weights():\\n    """Per class: original rows keep (1 - EXTRA_SHARE) of the class weight, the class\\\'s extra rows share EXTRA_SHARE of it."""\\n    n_tr, n_ex = np.bincount(y, minlength=K).astype(float), np.bincount(EX_Y, minlength=K).astype(float)\\n    w_tr = np.where(n_ex[y] > 0, 1.0 - EXTRA_SHARE, 1.0)\\n    w_ex = EXTRA_SHARE * n_tr[EX_Y] / np.maximum(n_ex[EX_Y], 1.0)\\n    return np.concatenate([w_tr, w_ex])\\n\\n\\ngrams = StyleGrams().fit(TR + EX_T)                       # vocabulary and idf from train + extra\\nX_tr, X_ex, X_te = grams.transform(TR), grams.transform(EX_T) if len(EX_T) else None, grams.transform(TE)\\n\\n\\ndef fit_svc(X, labels, C=0.5, **kw):\\n    return LinearSVC(C=C, random_state=42, max_iter=4000).fit(X, labels, **kw)\\n\\n\\nif X_ex is not None:\\n    svc_plain = fit_svc(vstack([X_tr, X_ex]).tocsr(), np.concatenate([y, EX_Y]), sample_weight=extra_sample_weights())\\nelse:\\n    svc_plain = fit_svc(X_tr, y)\\nP_style = svc_probs(svc_plain, X_te)\\n\\n# centred twin: prompt-centred training rows (extra rows centred within their own prompts) -> inferred test groups\\nX_tr_c = sparse_group_centre(X_tr, PROMPT)\\nif X_ex is not None:\\n    svc_centred = fit_svc(vstack([X_tr_c, sparse_group_centre(X_ex, EX_G, GROUP_MIN, 10 ** 6)]).tocsr(), np.concatenate([y, EX_Y]),\\n                          sample_weight=extra_sample_weights())\\nelse:\\n    svc_centred = fit_svc(X_tr_c, y)\\nG_style = normalised_content_groups(TE)\\nrows_style = eligible(G_style)\\ndescribe_groups(\\\'stylegram groups\\\', G_style)\\nif rows_style.any():\\n    P_style[rows_style] = svc_probs(svc_centred, sparse_group_centre(X_te[rows_style], G_style[rows_style]))\\nif TINY:   # exercise the centred path on training rows grouped by their true prompt (output discarded)\\n    st_idx, st_g = self_test_rows()\\n    acc = np.mean(svc_centred.classes_[svc_centred.decision_function(sparse_group_centre(X_tr[st_idx], st_g)).argmax(1)] == y[st_idx])\\n    print(f\\\'  centred stylegram self-test on {len(st_idx)} training rows (seen in training): acc {acc:.2f}\\\', flush=True)\\nadd_member(\\\'stylegram\\\', P_style)\\ndel X_tr_c, svc_plain, svc_centred, P_style\\n\\n\\n# ---- gates: style SVCs that only re-rank inside one family after the blend\\ndef family_gate(texts, labels, classes, C, sample_weight=None, balanced=False):\\n    g = StyleGrams().fit(texts)\\n    svc = LinearSVC(C=C, class_weight=\\\'balanced\\\' if balanced else None, random_state=42, max_iter=4000).fit(\\n        g.transform(texts), labels, sample_weight=sample_weight)\\n    assert list(svc.classes_) == list(classes)\\n    d = svc.decision_function(g.transform(TE))\\n    return np.column_stack([-d, d]) if len(classes) == 2 else d\\n\\n\\nGATES = {}\\napi_claude = sorted(glob.glob(\\\'/kaggle/input/**/ntl_extra_passages_claude.csv\\\', recursive=True))\\nif api_claude:\\n    api = pd.read_csv(api_claude[0])\\n    m = np.isin(y, FAMILY_IDX[\\\'claude\\\'])\\n    GATES[\\\'claude\\\'] = family_gate([t for t, k in zip(TR, m) if k] + api[\\\'text\\\'].astype(str).tolist(),\\n                                  np.concatenate([y[m], api[\\\'model\\\'].map(LAB2I).to_numpy(dtype=int)]), FAMILY_IDX[\\\'claude\\\'], 0.5)\\n    print(f\\\'claude gate: {int(m.sum())} official + {len(api)} API rows\\\', flush=True)\\n    del api\\nelse:\\n    print(\\\'Claude API passages not attached: Claude gate off\\\', flush=True)\\nfor fam, C in [(\\\'gpt\\\', 1.0), (\\\'gemini\\\', 1.0), (\\\'glm\\\', 0.5)]:\\n    m, me = np.isin(y, FAMILY_IDX[fam]), np.isin(EX_Y, FAMILY_IDX[fam])\\n    GATES[fam] = family_gate([t for t, k in zip(TR, m) if k] + [t for t, k in zip(EX_T, me) if k], np.concatenate([y[m], EX_Y[me]]),\\n                             FAMILY_IDX[fam], C, sample_weight=np.concatenate([np.ones(int(m.sum())), np.full(int(me.sum()), 0.25)]), balanced=True)\\n    print(f\\\'{fam} gate: {int(m.sum())} official + {int(me.sum())} extra rows\\\', flush=True)\\ndel grams, X_tr, X_ex, X_te\\n\\n# ============================================================ 3. stylometric features + LightGBM\\nPUNCT = list(\\\',.;:!?()[]"\\\\\\\'-\\\') + [\\\'—\\\', \\\'–\\\', \\\'‘\\\', \\\'’\\\', \\\'“\\\', \\\'”\\\', \\\'…\\\', \\\'*\\\', \\\'/\\\', \\\'%\\\']\\nSTOPS = sorted(ENGLISH_STOP_WORDS)\\nSENT_RE, WORD_RE = re.compile(r\\\'(?<=[.!?])\\\\s+\\\'), re.compile(r"[A-Za-z’\\\']+")\\n\\n\\ndef stylometry(t):\\n    words = WORD_RE.findall(t)\\n    low = [w.lower() for w in words]\\n    nw, nc = max(len(words), 1), max(len(t), 1)\\n    sents = [s for s in SENT_RE.split(t.strip()) if s]\\n    sl = np.array([len(WORD_RE.findall(s)) for s in sents] or [0], dtype=float)\\n    wl = np.array([len(w) for w in words] or [0], dtype=float)\\n    f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(), (wl >= 9).mean(), len(set(low)) / nw,\\n         sum(c.isupper() for c in t) / nc, sum(c.isdigit() for c in t) / nc, sum(ord(c) > 127 for c in t) / nc,\\n         t.count(\\\', \\\') / nw, t.count(\\\' and \\\') / nw, t.count(\\\' but \\\') / nw,\\n         sum(1 for s in sents if s[:1].isupper() and s.split()[0].lower() in (\\\'this\\\', \\\'these\\\', \\\'it\\\', \\\'they\\\')) / max(len(sents), 1),\\n         sum(w.endswith(\\\'ly\\\') for w in words) / nw, sum(w.endswith(\\\'ing\\\') for w in words) / nw, sum(w.endswith(\\\'tion\\\') for w in words) / nw]\\n    f += [t.count(q) / nw for q in PUNCT]\\n    counts = pd.Series(low).value_counts() if low else pd.Series(dtype=float)\\n    f += [counts.get(s, 0) / nw for s in STOPS]\\n    return f\\n\\n\\nF_tr, F_te = (np.array([stylometry(t) for t in txt], dtype=np.float32) for txt in (TR, TE))\\ntry:\\n    import lightgbm as lgb\\n    booster = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=10, subsample=0.8, subsample_freq=1,\\n                                 colsample_bytree=0.5, reg_lambda=1.0, random_state=0, verbose=-1, n_jobs=4).fit(F_tr, y)\\n    add_member(\\\'stylometry_lgb\\\', full_prob(booster.predict_proba(F_te), booster.classes_))\\n    del booster\\nexcept ImportError:\\n    assert SMOKE, \\\'lightgbm missing\\\'\\n    print(\\\'smoke: lightgbm unavailable, stylometry member skipped\\\', flush=True)\\ndel F_tr, F_te\\n\\n# ============================================================ 4b. shared-prompt Gaussian group head\\n# Verbatim core of our research tool tools/group_gaussian.py (sha256 8a42427b77a37f56...), numpy only.\\n# Model per dense view, in a reduced space z (standardise -> within-prompt-residual PCA, fitted on training rows):\\n#   z_i = mu_{y_i} + p_g + e_i,  p_g ~ N(0, S_p) shared by every passage of prompt group g,  e_i ~ N(0, S_e).\\n# A joint diagonalisation makes the dimensions independent; for a detected group of n passages the exact joint\\n# log-likelihood of a label tuple is sum_i a_i(c_i) + sum_{i<j} b_ij(c_i, c_j). Groups of 2-3 are decoded exactly over\\n# all label tuples with the at-most-once constraint; larger groups by a cavity EM with the finite-population-corrected\\n# prompt-effect estimate lam / (1 + (n-1) lam); singletons integrate the prompt effect out (never used here).\\nimport itertools\\nK = 18\\nDEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,\\n           "iters": 5, "repel": True, "sp_scale": 1.0}\\n\\n# =================================================================== primitives ====\\ndef _lse(A, axis=None, keepdims=False):\\n    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)\\n    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m\\n    if keepdims: return out\\n    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())\\n\\ndef log_softmax(L): return L - _lse(L, axis=1, keepdims=True)\\n\\ndef _lw_shrinkage(X):\\n    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""\\n    from sklearn.covariance import ledoit_wolf_shrinkage\\n    return float(ledoit_wolf_shrinkage(X, assume_centered=True))\\n\\ndef _shrunk(S, a):\\n    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)\\n\\ndef _two_way(Z, y, pinv, G, iters=4):\\n    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""\\n    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)\\n    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))\\n    for _ in range(iters):\\n        R = Z - P[pinv]\\n        for c in cls: mu[c] = R[y == c].mean(0)\\n        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]\\n    return mu, P, cnt_p\\n\\ndef fit(X, y, prompts, cfg=None):\\n    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.\\n    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller\\\'s cross-fit)."""\\n    c = dict(DEFAULT); c.update(cfg or {})\\n    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape\\n    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)\\n    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx\\n    if c["red"] == "none":\\n        V = np.eye(D)\\n    else:\\n        if c["red"] == "wpca":\\n            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]\\n        elif c["red"] == "pca": R = Xs\\n        else: raise ValueError(c["red"])\\n        d = int(min(c["d"], N - G - 1, D))\\n        # economy SVD through the (N x N) Gram matrix (N << D for the real views)\\n        if N < D:\\n            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]\\n            V = (R.T @ U) / np.sqrt(w)\\n        else:\\n            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T\\n    Z = Xs @ V; d = Z.shape[1]\\n    mu, P, ng = _two_way(Z, y, pinv, G)\\n    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)\\n    Se_raw = E.T @ E / dof\\n    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)\\n    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw\\n    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])\\n    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]\\n    # joint diagonalisation: A Se A\\\' = I, A Sp A\\\' = diag(lam)\\n    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)\\n    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li\\n    W = V @ A.T\\n    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,\\n            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}\\n\\ndef transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]\\n\\n_MASK = {}\\ndef _inj_mask(n, C):\\n    if (n, C) not in _MASK:\\n        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)\\n        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]\\n        _MASK[(n, C)] = bad\\n    return _MASK[(n, C)]\\n\\ndef singleton_ll(U, m, lam):\\n    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""\\n    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)\\n\\ndef joint_terms(U, m, lam):\\n    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""\\n    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]\\n    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)\\n    return a, B\\n\\ndef joint_table(a, B):\\n    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""\\n    n, C = a.shape; Q = np.zeros([C] * n)\\n    for i in range(n):\\n        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)\\n    for i, j in itertools.combinations(range(n), 2):\\n        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)\\n    return Q\\n\\ndef exact_marginals(Q, T=1.0, inj=True):\\n    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""\\n    n = Q.ndim; C = Q.shape[0]; Qt = Q / T\\n    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)\\n    lz = _lse(Qt); out = np.empty((n, C))\\n    for i in range(n):\\n        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz\\n    return out\\n\\ndef em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):\\n    """cavity EM for one group (any n >= 2): returns (n, C) log q."""\\n    n = len(U); v = lam / (1 + (n - 1) * lam)\\n    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))\\n    q = np.exp(log_softmax(L))\\n    for _ in range(iters):\\n        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)\\n        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T\\n        if repel:   # mean-field at-most-once: discourage classes the other members already hold\\n            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)\\n        q = np.exp(log_softmax(L))\\n    return log_softmax(L)\\n\\nLAM_DIFF = 1e6\\ndef lam_g(prm, cfg=None):\\n    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg \\\'diff\\\'): then the joint likelihood\\n    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs\\n    centred prototypes) and the prompt term cancels exactly."""\\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})\\n    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]\\n\\ndef group_logpost(U, prm, T1=None, Tg=None, cfg=None):\\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)\\n    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg\\n    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)\\n    if n <= c["n_exact"]:\\n        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])\\n    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])\\n\\ndef scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):\\n    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""\\n    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))\\n    sizes = np.bincount(lab); s1 = sizes[lab] == 1\\n    T1 = prm["T1"] if T1 is None else T1\\n    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)\\n    for g in np.where(sizes >= 2)[0]:\\n        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)\\n    return out\\n\\n# ============================================================ 4. frozen models: plain heads, centred twins, Gaussian group heads\\nif not SMOKE:\\n    from transformers import AutoConfig, AutoModel, AutoTokenizer\\nCACHED_PARTS = {}   # key -> {part name: (train features, test features)}\\n\\n\\ndef find_checkpoint(model_type, hidden=None, dir_name=None):\\n    """A checkpoint directory under /kaggle/input by model_type (+ hidden size) or by directory name."""\\n    pattern = f\\\'/kaggle/input/**/{dir_name}/config.json\\\' if dir_name else \\\'/kaggle/input/**/config.json\\\'\\n    for cfg in sorted(glob.glob(pattern, recursive=True)):\\n        try:\\n            c = json.load(open(cfg))\\n        except Exception:\\n            continue\\n        if dir_name or (c.get(\\\'model_type\\\') == model_type and (hidden is None or c.get(\\\'hidden_size\\\') == hidden)):\\n            return os.path.dirname(cfg)\\n    return None\\n\\n\\ndef load_truncated_decoder(path, keep, dtype):\\n    cfg = AutoConfig.from_pretrained(path)\\n    cfg.num_hidden_layers = keep\\n    if getattr(cfg, \\\'layer_types\\\', None):\\n        cfg.layer_types = cfg.layer_types[:keep]\\n    kw = dict(config=cfg, dtype=dtype)\\n    if DEV == \\\'cuda\\\':\\n        kw[\\\'device_map\\\'] = \\\'auto\\\'\\n    return AutoModel.from_pretrained(path, **kw).eval()\\n\\n\\ndef style_token_mask(tok):\\n    """Vocabulary mask of \\\'style\\\' tokens: the decoded piece is an English stop word or pure punctuation."""\\n    pieces = [tok.decode([i]) for i in range(len(tok))]\\n    return torch.tensor([(p.strip().lower() in ENGLISH_STOP_WORDS) or bool(re.fullmatch(r\\\'\\\\s*[^\\\\w\\\\s]+\\\\s*\\\', p)) for p in pieces])\\n\\n\\n@torch.no_grad()\\ndef pooled_states(model, tok, texts, parts, decoder, bs, max_len=512):\\n    """{\\\'L{layer}_{pool}\\\': (n, hidden)} masked mean pools of hidden_states[layer] (all tokens, or style tokens only)."""\\n    H = model.config.hidden_size\\n    out = {f\\\'L{L}_{pool}\\\': np.zeros((len(texts), H), dtype=np.float32) for L, pool in parts}\\n    smask = style_token_mask(tok) if any(pool == \\\'style\\\' for _, pool in parts) else None\\n    dev = next(model.parameters()).device\\n    order = np.argsort([len(t) for t in texts])\\n    retried = 0\\n    for b0 in range(0, len(texts), bs):\\n        idx = order[b0:b0 + bs]\\n        if decoder:\\n            batch = [(tok.bos_token or \\\'\\\') + texts[i] for i in idx]\\n            enc = tok(batch, return_tensors=\\\'pt\\\', padding=True, truncation=True, max_length=max_len, add_special_tokens=False).to(dev)\\n        else:\\n            enc = tok([texts[i] for i in idx], return_tensors=\\\'pt\\\', padding=True, truncation=True, max_length=max_len).to(dev)\\n        hs = model(**enc, output_hidden_states=True).hidden_states\\n        if decoder and not all(torch.isfinite(hs[L]).all() for L, _ in parts):   # fp16 overflow: redo the batch in bf16\\n            retried += 1\\n            model.to(torch.bfloat16)\\n            hs = model(**enc, output_hidden_states=True).hidden_states\\n        m_all = enc[\\\'attention_mask\\\'].float()\\n        for L, pool in parts:\\n            m = m_all * smask.to(m_all.device)[enc[\\\'input_ids\\\']].float() if pool == \\\'style\\\' else m_all\\n            m = m.unsqueeze(-1).to(hs[L].device)\\n            out[f\\\'L{L}_{pool}\\\'][idx] = ((hs[L].float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()\\n        del hs, enc\\n    if retried:\\n        print(f\\\'  {retried} batches redone in bf16\\\', flush=True)\\n    return {k: np.nan_to_num(v) for k, v in out.items()}\\n\\n\\ndef pseudo_parts(parts, dim=48):\\n    """SMOKE only: deterministic stand-ins for the model passes (TF-IDF SVD of the texts), same downstream code."""\\n    from sklearn.decomposition import TruncatedSVD\\n    tf = TfidfVectorizer(analyzer=\\\'char\\\', ngram_range=(2, 4), sublinear_tf=True, min_df=2).fit(TR + TE)\\n    out = {}\\n    for j, (L, pool) in enumerate(parts):\\n        svd = TruncatedSVD(dim, random_state=L + 7 * j).fit(tf.transform(TR))\\n        out[f\\\'L{L}_{pool}\\\'] = (svd.transform(tf.transform(TR)).astype(np.float32), svd.transform(tf.transform(TE)).astype(np.float32))\\n    return out\\n\\n\\ndef extract_parts(key, parts, loader):\\n    """Train and test pools for every requested part; loader() returns (model, tok, decoder flag, batch size)."""\\n    if SMOKE:\\n        return pseudo_parts(parts)\\n    model, tok, decoder, bs = loader()\\n    tr, te = pooled_states(model, tok, TR, parts, decoder, bs), pooled_states(model, tok, TE, parts, decoder, bs)\\n    del model, tok\\n    gc.collect()\\n    if DEV == \\\'cuda\\\':\\n        torch.cuda.empty_cache()\\n    return {k: (tr[k], te[k]) for k in tr}\\n\\n\\ndef decoder_loader(model_type, hidden, keep):\\n    def load():\\n        path = find_checkpoint(model_type, hidden)\\n        assert path, f\\\'no {model_type} checkpoint with hidden size {hidden} under /kaggle/input\\\'\\n        tok = AutoTokenizer.from_pretrained(path)\\n        tok.padding_side = \\\'right\\\'\\n        if tok.pad_token is None:\\n            tok.pad_token = tok.eos_token\\n        return load_truncated_decoder(path, keep, torch.float16 if DEV == \\\'cuda\\\' else torch.float32), tok, True, 8\\n    return load\\n\\n\\ndef encoder_loader(dir_name):\\n    def load():\\n        path = find_checkpoint(None, dir_name=dir_name)\\n        assert path, f\\\'encoder directory {dir_name} not found under /kaggle/input\\\'\\n        return AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval(), AutoTokenizer.from_pretrained(path), False, 16\\n    return load\\n\\n\\ndef concat_parts(cache, parts):\\n    return (np.concatenate([cache[f\\\'L{L}_{p}\\\'][0] for L, p in parts], 1), np.concatenate([cache[f\\\'L{L}_{p}\\\'][1] for L, p in parts], 1))\\n\\n\\ndef plain_and_twin(key, Etr, Ete, C):\\n    """Plain head (test-side standardisation) for the blend; centred twin evidence on grouped rows."""\\n    if W[key] > 0:\\n        sc = StandardScaler().fit(Etr)\\n        clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(Etr), y)\\n        Xte = (Ete - Ete.mean(0)) / (Ete.std(0) + 1e-6) if len(Ete) >= 2 else sc.transform(Ete)\\n        add_member(key, full_prob(clf.predict_proba(Xte), clf.classes_))\\n    else:\\n        print(f\\\'  {key}: plain head has weight 0 in the 0.563 blend, not fitted\\\', flush=True)\\n    if key in TWIN_W:\\n        csc = StandardScaler().fit(group_centre_dense(Etr, PROMPT))\\n        twin = LogisticRegression(C=C, max_iter=5000).fit(csc.transform(group_centre_dense(Etr, PROMPT)), y)\\n        if GROUP_ROWS.any():\\n            pc = twin.predict_proba(csc.transform(group_centre_dense(Ete[GROUP_ROWS], G_TE[GROUP_ROWS])))\\n            add_member(key + \\\'_twin\\\', full_prob(pc, twin.classes_), rows=GROUP_ROWS, weight=TWIN_W[key])\\n        if TINY:\\n            st_idx, st_g = self_test_rows()\\n            ps = twin.predict_proba(csc.transform(group_centre_dense(Etr[st_idx], st_g)))\\n            print(f\\\'  {key} twin self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(twin.classes_[ps.argmax(1)] == y[st_idx]):.2f}\\\', flush=True)\\n\\n\\ndef gaussian_group_head(name, spec, Xtr, Xte):\\n    """Our addition: fit the shared-prompt Gaussian on the training prompts, score the grouped test rows jointly within their group."""\\n    prm = fit(Xtr, y, PROMPT, dict(GAUSS_SETTING, d=spec[\\\'d\\\']))\\n    prm[\\\'T1\\\'], prm[\\\'Tg\\\'] = spec[\\\'T1\\\'], spec[\\\'Tg\\\']\\n    print(f\\\'  {name}: d {prm["info"]["d"]}, shrinkage e/p {prm["info"]["a_e"]}/{prm["info"]["a_p"]}, lam pct {prm["info"]["lam_pct"]}\\\', flush=True)\\n    if GROUP_ROWS.any():\\n        LP = scores(Xte[GROUP_ROWS], np.unique(G_TE[GROUP_ROWS], return_inverse=True)[1], prm)\\n        add_member(name, np.exp(LP), rows=GROUP_ROWS, weight=spec[\\\'w\\\'])\\n    if TINY:\\n        st_idx, st_g = self_test_rows()\\n        LP = scores(Xtr[st_idx], st_g, prm)\\n        print(f\\\'  {name} self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(LP.argmax(1) == y[st_idx]):.2f}\\\', flush=True)\\n\\n\\nfor key, model_type, hidden, parts, extra_parts, C in LM_HEADS:\\n    print(f\\\'[{key}] {model_type} hidden {hidden} parts {parts + extra_parts}\\\', flush=True)\\n    cache = extract_parts(key, parts + extra_parts, decoder_loader(model_type, hidden, max(L for L, _ in parts + extra_parts) + 1))\\n    if G_TE is None:   # the GLM pass comes first: its layer-24 pool defines the test groups\\n        G_TE = mixed_groups(TE, cache[f\\\'L{GROUP_LAYER}_mean\\\'][1])\\n        GROUP_ROWS = eligible(G_TE)\\n        describe_groups(\\\'test groups (content + GLM L24)\\\', G_TE)\\n        if SMOKE:\\n            same_g, same_p = G_TE[:, None] == G_TE[None, :], test[\\\'prompt_id\\\'].to_numpy(dtype=object)[:, None] == test[\\\'prompt_id\\\'].to_numpy(dtype=object)[None, :]\\n            off = ~np.eye(len(TE), dtype=bool)\\n            print(f\\\'  smoke grouping: pair precision {(same_g & same_p & off).sum() / max((same_g & off).sum(), 1):.2f} \\\'\\n                  f\\\'recall {(same_g & same_p & off).sum() / max((same_p & off).sum(), 1):.2f}\\\', flush=True)\\n    CACHED_PARTS[key] = cache\\n    plain_and_twin(key, *concat_parts(cache, parts), C)\\n    for name, spec in GAUSS_HEADS.items():\\n        if spec[\\\'src\\\'] == key:\\n            gaussian_group_head(name, spec, *cache[spec[\\\'part\\\']])\\n    del cache\\n    CACHED_PARTS.pop(key)\\n    gc.collect()\\n\\nfor key, dir_name, parts, C in ENC_HEADS:\\n    print(f\\\'[{key}] {dir_name} parts {parts}\\\', flush=True)\\n    cache = extract_parts(key, parts, encoder_loader(dir_name))\\n    plain_and_twin(key, *concat_parts(cache, parts), C)\\n    for name, spec in GAUSS_HEADS.items():\\n        if spec[\\\'src\\\'] == key:\\n            gaussian_group_head(name, spec, *cache[spec[\\\'part\\\']])\\n    del cache\\n    gc.collect()\\n\\n# ============================================================ 5. LoRA classifiers (inference only)\\n@torch.no_grad()\\ndef score_lora_checkpoints(trunk_spec, pool_layers, checkpoints):\\n    """One truncated trunk, adapters swapped in turn. Plain readout -> blend member; centred readout -> twin on grouped rows;\\n    within-Claude readout -> Claude gate."""\\n    import importlib.util as iu\\n    _find = iu.find_spec   # Kaggle\\\'s torchao build is rejected by peft; hide it (unused here)\\n    iu.find_spec = lambda name, *a, **k: None if name.split(\\\'.\\\')[0] == \\\'torchao\\\' else _find(name, *a, **k)\\n    from peft import PeftModel\\n    import peft.import_utils as pi\\n    pi.is_torchao_available = lambda *a, **k: False\\n    from safetensors.torch import load_file\\n    model_type, H, keep = trunk_spec\\n    base = find_checkpoint(model_type, H)\\n    assert base, f\\\'no {model_type} base checkpoint under /kaggle/input\\\'\\n    cfg = AutoConfig.from_pretrained(base)\\n    cfg.num_hidden_layers, cfg.use_cache = keep, False\\n    if getattr(cfg, \\\'layer_types\\\', None) is not None:\\n        cfg.layer_types = cfg.layer_types[:keep]\\n    trunk = AutoModel.from_pretrained(base, config=cfg, dtype=torch.float16, device_map=\\\'auto\\\').eval()\\n    for key, marker, norms_file, anchor_file in checkpoints:\\n        hits = sorted(glob.glob(\\\'/kaggle/input/\\\' + marker, recursive=True))\\n        assert hits, f\\\'checkpoint for {key} not attached ({marker})\\\'\\n        ck = pathlib.Path(os.path.dirname(hits[0]))\\n        print(f\\\'[{key}] {ck}\\\', flush=True)\\n        tok = AutoTokenizer.from_pretrained(ck / \\\'tokenizer\\\')\\n        tok.padding_side = \\\'right\\\'\\n        if tok.pad_token is None:\\n            tok.pad_token = tok.eos_token\\n        lm = PeftModel.from_pretrained(trunk, ck / \\\'adapter\\\').eval()\\n        state = load_file(str(ck / norms_file))\\n        norms = torch.nn.ModuleList([torch.nn.LayerNorm(H) for _ in pool_layers])\\n        norms.load_state_dict({k[len(\\\'norms.\\\'):]: v for k, v in state.items() if k.startswith(\\\'norms.\\\')})\\n        ro = np.load(ck / \\\'readouts.npz\\\')\\n        assert list(ro[\\\'classes\\\']) == LABELS and list(ro[\\\'pool_layers\\\']) == pool_layers\\n        dev = lm.get_input_embeddings().weight.device\\n        norms = norms.to(dev).float()\\n\\n        def token_ids(texts):\\n            return [ids + [tok.eos_token_id] for ids in tok(list(texts), add_special_tokens=False, truncation=True, max_length=LORA_MAX_LEN)[\\\'input_ids\\\']]\\n\\n        def features(texts, bs):\\n            ids = token_ids(texts)\\n            Z = np.zeros((len(ids), H * len(pool_layers)), dtype=np.float32)\\n            order = np.argsort([len(t) for t in ids], kind=\\\'stable\\\')\\n            for b0 in range(0, len(order), bs):\\n                idx = order[b0:b0 + bs]\\n                batch = tok.pad({\\\'input_ids\\\': [ids[i] for i in idx]}, return_tensors=\\\'pt\\\').to(dev)\\n                with torch.autocast(\\\'cuda\\\', dtype=torch.float16):\\n                    hs = lm(**batch, output_hidden_states=True, use_cache=False).hidden_states\\n                m = batch[\\\'attention_mask\\\'].unsqueeze(-1).float()\\n                m[:, 0] = 0   # the first token is masked out of the mean, as in training\\n                Z[idx] = torch.cat([nm((hs[L].float() * m.to(hs[L].device)).sum(1).to(dev) / m.sum(1).to(dev))\\n                                    for L, nm in zip(pool_layers, norms)], -1).cpu().numpy()\\n                del hs, batch\\n            return Z\\n\\n        if (ck / anchor_file).exists():   # the checkpoint\\\'s own features for training rows 0..2: our forward pass must reproduce them\\n            saved = np.load(ck / anchor_file)[\\\'features\\\'].astype(np.float32)\\n            ours = features(TR[:3], 3)\\n            rel = float(np.abs(ours - saved).mean() / (np.abs(saved).mean() + 1e-9))\\n            print(f\\\'  parity with the checkpoint anchors: max abs {float(np.abs(ours - saved).max()):.4f}, mean rel {rel:.5f}\\\', flush=True)\\n            assert rel < 0.02, f\\\'{key}: our features do not match the checkpoint\\\'\\n        Z = features(TE, LORA_BS)\\n        add_member(key, softmax_rows(Z @ ro[\\\'plain_coef\\\'].T + ro[\\\'plain_intercept\\\']))\\n        if \\\'claude_coef\\\' in ro.files:\\n            lc = Z @ ro[\\\'claude_coef\\\'].T + ro[\\\'claude_intercept\\\']\\n            GATES[\\\'claude_lora\\\'] = lc - np.log(np.exp(lc - lc.max(1, keepdims=True)).sum(1, keepdims=True)) - lc.max(1, keepdims=True)\\n        if key in LORA_TWIN_W and GROUP_ROWS.any():\\n            Zc = group_centre_dense(Z[GROUP_ROWS], G_TE[GROUP_ROWS])\\n            add_member(key + \\\'_twin\\\', softmax_rows(Zc @ ro[\\\'center_coef\\\'].T + ro[\\\'center_intercept\\\']), rows=GROUP_ROWS, weight=LORA_TWIN_W[key])\\n        trunk = lm.unload()\\n        del lm, Z\\n        gc.collect()\\n        torch.cuda.empty_cache()\\n    del trunk\\n    gc.collect()\\n    torch.cuda.empty_cache()\\n\\n\\nif not SMOKE:\\n    for trunk_spec, pool_layers, checkpoints in LORA_RUNS:\\n        score_lora_checkpoints(trunk_spec, pool_layers, checkpoints)\\n\\n# ============================================================ 6. token-attention heads + UltraBERT\\nclass TokenAttention(torch.nn.Module if torch is not None else object):   # the module of the public training kernels (lucasboesen/name-that-llm-train-tokattn, -tokattn2)\\n    def __init__(self, k, h=4096):\\n        super().__init__()\\n        self.drop, self.proj, self.score, self.out = torch.nn.Dropout(0.2), torch.nn.Linear(h, 64), torch.nn.Linear(64, 1), torch.nn.Linear(128, k)\\n\\n    def forward(self, x, m):\\n        h = torch.tanh(self.proj(self.drop(x)))\\n        a = torch.softmax(self.score(h).squeeze(-1).masked_fill(~m, -1e4), -1).unsqueeze(-1)\\n        mf = m.unsqueeze(-1).float()\\n        return self.out(self.drop(torch.cat([(h * mf).sum(1) / mf.sum(1).clamp(min=1), (a * h).sum(1)], -1)))\\n\\n\\n@torch.no_grad()\\ndef token_attention_probs(texts, head_file, backbone, bs=8):\\n    hits = sorted(glob.glob(\\\'/kaggle/input/**/\\\' + head_file, recursive=True))\\n    assert hits, f\\\'training-kernel output holding {head_file} not attached\\\'\\n    d = torch.load(hits[0], map_location=\\\'cpu\\\')\\n    assert list(d[\\\'classes\\\']) == LABELS\\n    decoder = d.get(\\\'decoder\\\', True)\\n    path = find_checkpoint(\\\'qwen3_5\\\') if backbone == \\\'qwen3_5\\\' else find_checkpoint(None, dir_name=\\\'deberta-v3-large\\\')\\n    assert path, f\\\'backbone {backbone} not attached\\\'\\n    cfg = AutoConfig.from_pretrained(path)\\n    if decoder:\\n        tc = getattr(cfg, \\\'text_config\\\', None) or cfg\\n        tc.num_hidden_layers = d[\\\'keep\\\']\\n        if getattr(tc, \\\'layer_types\\\', None):\\n            tc.layer_types = tc.layer_types[:d[\\\'keep\\\']]\\n        lm = AutoModel.from_pretrained(path, config=cfg, dtype=torch.float16, device_map=\\\'auto\\\').eval()\\n    else:\\n        lm = AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval()\\n    tok = AutoTokenizer.from_pretrained(path)\\n    tok.padding_side = \\\'right\\\'\\n    if tok.pad_token is None:\\n        tok.pad_token = tok.eos_token\\n    H = d.get(\\\'hidden\\\', 4096)\\n    net = TokenAttention(K, H)\\n    net.load_state_dict(d[\\\'state_dict\\\'])\\n    dev = lm.get_input_embeddings().weight.device\\n    net, mu, sd = net.to(dev).eval(), d[\\\'mu\\\'].float().to(dev), d[\\\'sd\\\'].float().to(dev)\\n    ids = tok(list(texts), add_special_tokens=not decoder, truncation=True, max_length=d[\\\'max_len\\\'])[\\\'input_ids\\\']\\n    P = np.zeros((len(texts), K))\\n    order = np.argsort([len(t) for t in ids])\\n    for b0 in range(0, len(order), bs):\\n        idx = order[b0:b0 + bs]\\n        batch = tok.pad({\\\'input_ids\\\': [ids[i] for i in idx]}, return_tensors=\\\'pt\\\').to(dev)\\n        out = lm(input_ids=batch[\\\'input_ids\\\'], attention_mask=batch[\\\'attention_mask\\\'], output_hidden_states=True)\\n        hs = (out.hidden_states if getattr(out, \\\'hidden_states\\\', None) is not None else out.language_model_outputs.hidden_states)[d[\\\'layer\\\']]\\n        n = batch[\\\'attention_mask\\\'].sum(1)\\n        T = max(int(n.max()) - 1, 1)\\n        x = torch.zeros(len(idx), T, H, device=dev)\\n        m = torch.zeros(len(idx), T, dtype=torch.bool, device=dev)\\n        for r in range(len(idx)):   # first token dropped, states rounded through fp16, as in training\\n            k = int(n[r]) - 1\\n            if k > 0:\\n                x[r, :k] = ((torch.nan_to_num(hs[r, 1:k + 1].to(dev).float()).half().float() - mu) / sd).clamp(-8, 8)\\n                m[r, :k] = True\\n        P[idx] = torch.softmax(net(x, m), -1).cpu().numpy()\\n        del hs, out, batch\\n    del lm\\n    gc.collect()\\n    torch.cuda.empty_cache()\\n    return P\\n\\n\\n@torch.no_grad()\\ndef ultrabert_probs(texts):\\n    from transformers import AutoModelForSequenceClassification\\n    folds = None\\n    for root in glob.glob(\\\'/kaggle/input/**/ultrabert-380m-ntl\\\', recursive=True):\\n        fs = [f for f in sorted(glob.glob(os.path.join(root, \\\'fold*\\\'))) if os.path.exists(os.path.join(f, \\\'config.json\\\'))\\n              and json.load(open(os.path.join(f, \\\'config.json\\\'))).get(\\\'model_type\\\') == \\\'ultrabert\\\']\\n        if len(fs) == 5:\\n            folds = fs\\n            break\\n    assert folds, \\\'ultrabert-380m-ntl (5 folds) not attached\\\'\\n    tok = AutoTokenizer.from_pretrained(folds[0], trust_remote_code=True)\\n    ids = tok(texts, truncation=True, max_length=512)[\\\'input_ids\\\']\\n    order = np.argsort([len(e) for e in ids])\\n    P = np.zeros((len(texts), K))\\n    for fp in folds:\\n        em = AutoModelForSequenceClassification.from_pretrained(fp, trust_remote_code=True, dtype=torch.float16, attn_implementation=\\\'sdpa\\\').to(DEV).eval()\\n        cols = [LAB2I[em.config.id2label[i]] for i in range(len(em.config.id2label))]\\n        for b0 in range(0, len(order), 32):\\n            idx = order[b0:b0 + 32]\\n            b = tok.pad({\\\'input_ids\\\': [ids[j] for j in idx]}, return_tensors=\\\'pt\\\').to(DEV)\\n            P[np.ix_(idx, cols)] += em(**b).logits.float().softmax(-1).cpu().numpy() / len(folds)\\n        del em\\n        gc.collect()\\n        torch.cuda.empty_cache()\\n    return P\\n\\n\\nif not SMOKE:\\n    for key, head_file, backbone in TOKATTN_HEADS:\\n        if TINY:   # in-sample check on training rows the head was fitted on: must be high, or the checkpoint is not being reproduced\\n            acc = float(np.mean(token_attention_probs(TR[:40], head_file, backbone).argmax(1) == y[:40]))\\n            print(f\\\'  {key} in-sample check on 40 training rows: acc {acc:.2f}\\\', flush=True)\\n            assert acc > 0.5, f\\\'{key} does not reproduce its training fit\\\'\\n        add_member(key, token_attention_probs(TE, head_file, backbone))\\n    add_member(\\\'ultrabert\\\', ultrabert_probs(TE))\\n\\n\'\n# Complementary feature ensemble: source Gaussian stack, no hard family gates.\n_new_labels = list(LABELS)\n_new_prob = softmax(LOGIT.copy())\n_old_namespace = {\'__name__\': \'__gaussian_stack__\'}\nexec(compile(_GAUSSIAN_SOURCE, \'public_gaussian_stack\', \'exec\'), _old_namespace)\nassert list(_old_namespace[\'test\'][\'id\'].astype(str)) == list(test[\'id\'].astype(str))\n_old_cols = [_old_namespace[\'LABELS\'].index(label) for label in _new_labels]\n_old_raw = _old_namespace[\'LOGIT\'][:, _old_cols]\n_old_prob = softmax(_old_raw)\n_mix_prob = 0.85 * _new_prob + 0.15 * _old_prob\nassert np.isfinite(_mix_prob).all() and np.allclose(_mix_prob.sum(1), 1.0)\nLOGIT = np.log(np.clip(_mix_prob, 1e-12, 1.0))\nprint(\'BLEND two-stage .85 + original four-Gaussian stack .15; labels aligned; single final Sinkhorn\', flush=True)\ndel _old_namespace, _old_raw, _old_prob, _new_prob, _mix_prob\nimport gc as _gc\n_gc.collect()\ntorch.cuda.empty_cache()\n\nP = softmax(LOGIT)\nif len(test) >= 50:   # skip on the 3-row public test.csv\n    target = np.full(K, len(P) / K)\n    for _ in range(100):\n        P *= target / P.sum(0)\n        P /= P.sum(1, keepdims=True)\nsub = pd.DataFrame({"id": test["id"].astype(str), "model": np.array(LABELS)[P.argmax(1)]})\nsub.to_parquet("submission.parquet", index=False)\nprint(sub["model"].value_counts().to_string())\nprint(f"done in {time.time() - T0:.0f}s")\n', 'numeric_transport.py': '"""No-pickle numeric transport; validate the bytes actually written."""\nfrom pathlib import Path\nimport numpy as np\n\ndef save_numeric(path, **arrays):\n    arrays={key:np.asarray(value) for key,value in arrays.items()}\n    assert all(not value.dtype.hasobject for value in arrays.values()), \'Object arrays forbidden\'\n    assert all(value.dtype.kind in \'USbiuf\' for value in arrays.values())\n    assert all(np.isfinite(value).all() for value in arrays.values() if value.dtype.kind in \'f\')\n    path=Path(path)\n    assert not path.exists(), \'No automatic overwrite or reuse\'\n    np.savez_compressed(path,**arrays)\n    recovered=load_numeric(path)\n    assert set(recovered)==set(arrays)\n    for key,value in arrays.items():\n        assert recovered[key].dtype==value.dtype\n        np.testing.assert_array_equal(recovered[key],value)\n\ndef load_numeric(path):\n    with np.load(path,allow_pickle=False) as saved:\n        arrays={key:saved[key].copy() for key in saved.files}\n    assert all(not value.dtype.hasobject for value in arrays.values())\n    return arrays\n', 'compute_olmo.py': '"""Frozen native24 OLMo2 plain readout; one query pass, no fitting."""\nimport hashlib,json,os,time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nfrom olmo_component import OLMoFinalPlain\n\nhere=Path(__file__).resolve().parent;config=json.loads((here/\'candidate-config.json\').read_text())\nassert config[\'source_ready\'] and config[\'OLMo_plain_raw_coefficient\']==.03\nspec=json.loads((here/\'olmo-spec.json\').read_text())\nassert spec[\'inference_authorized\'] and spec[\'root_production_admitted\'] and spec[\'public_asset_release_audited\']\ndef sha(p):\n    with p.open(\'rb\') as f:return hashlib.file_digest(f,\'sha256\').hexdigest()\nassets=[p for p in Path(\'/kaggle/input\').rglob(\'asset_manifest.json\') if sha(p)==spec[\'compact_manifest_sha256\']]\nbase_config=next(r for r in spec[\'base_files\'] if r[\'name\']==\'config.json\')\nbases=[p for p in Path(\'/kaggle/input\').rglob(\'config.json\') if sha(p)==base_config[\'sha256\']]\n\ntests=[p/\'test.csv\' for p in (Path(\'/kaggle/input/competitions/name-that-llm\'),Path(\'/kaggle/input/name-that-llm\')) if (p/\'test.csv\').is_file()]\nif os.environ.get(\'NTL_QUERY_CSV\'): tests=[Path(os.environ[\'NTL_QUERY_CSV\'])]\nassert len(assets)==len(bases)==len(tests)==1\ncache=Path(os.environ[\'NTL_CANDIDATE_TEMP\']).resolve();assert cache.is_relative_to(Path(\'/kaggle/temp\').resolve())\nassert not (cache/\'olmo_plain.npz\').exists(),\'No automatic repeat/reuse\'\nquery=pd.read_csv(tests[0],usecols=[\'id\',\'text\'],dtype=str)\nquery[\'text\']=query[\'text\'].fillna(\'\').astype(str)\nids=query.id.to_numpy(dtype=str);labels=np.asarray(spec[\'classes\'],dtype=str)\nnp.testing.assert_array_equal(query.id.to_numpy(dtype=str),ids)\nassert os.environ[\'CUDA_VISIBLE_DEVICES\'].startswith(\'GPU-\')\nstarted=time.monotonic()\nmodel=OLMoFinalPlain(here/\'olmo-spec.json\',assets[0].parent,bases[0].parent,None)\nnp.testing.assert_array_equal(model.readouts[\'classes\'],labels)\nfrom worker_barrier import ready_and_wait\nready_and_wait(\'olmo\')\nimport torch\nassert torch.cuda.device_count()==1\nbegin,end=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True)\nbegin.record();result=model.predict_plain_logp(ids,query.text.tolist());end.record();torch.cuda.synchronize()\nevent_ms=float(begin.elapsed_time(end));assert event_ms>0\nnp.testing.assert_array_equal(result[\'ids\'],ids);np.testing.assert_array_equal(result[\'classes\'],labels)\nassert result[\'plain_logp\'].dtype==np.float32 and result[\'plain_logp\'].shape==(len(ids),18) and np.isfinite(result[\'plain_logp\']).all()\nassert torch.cuda.max_memory_reserved()<=13*1024**3\nfrom numeric_transport import save_numeric\nsave_numeric(cache/\'olmo_plain.npz\',ids=ids,labels=labels,plain_logp=result[\'plain_logp\'])\nsave_numeric(Path(\'/kaggle/working/olmo_plain_recovery.npz\'),ids=ids,labels=labels,plain_logp=result[\'plain_logp\'])\nPath(\'/kaggle/working/olmo_inference_audit.json\').write_text(json.dumps(dict(status=\'FROZEN_OLMO2_NATIVE24_PLAIN_QUERY_COMPLETE\',rows=len(ids),feature_width=24576,plain_dtype=\'float32\',classes=labels.tolist(),original_model_first24_final_RMSNorm=True,pool_layers=[4,8,12,16,20,24],synthetic_anchor_replay=model.anchor_replay,CUDA_VISIBLE_DEVICES=os.environ[\'CUDA_VISIBLE_DEVICES\'],query_cuda_event_ms=event_ms,peak_reserved_bytes=torch.cuda.max_memory_reserved(),wall_seconds=time.monotonic()-started,model_fits=0),indent=2))\n', 'olmo_component.py': '"""Portable trained final450 OLMo2 native first24 plain readout; no fitting."""\nimport hashlib,importlib.metadata,json,os\nfrom pathlib import Path\ndef sha(path):\n    with Path(path).open(\'rb\') as stream:return hashlib.file_digest(stream,\'sha256\').hexdigest()\ndef validate_contract(spec,artifact_root,base_root,diagnostic_only=False):\n    assert not diagnostic_only and spec[\'inference_authorized\'] and spec[\'root_production_admitted\']\n    root,base=Path(artifact_root),Path(base_root)\n    assert sha(root/\'asset_manifest.json\')==spec[\'compact_manifest_sha256\']\n    manifest=json.loads((root/\'asset_manifest.json\').read_text())\n    assert manifest[\'status\']==\'PUBLIC_SAFE_OLMO2_NATIVE24_FINAL_INFERENCE\'\n    for flag in (\'contains_canonical_competition_IDs\',\'contains_competition_text_or_token_IDs\',\'contains_targets\',\'contains_optimizer_or_RNG\',\'contains_training_or_query_features\'):\n        assert manifest[flag] is False\n    assert {r[\'relative_path\'] for r in manifest[\'files\']}==set(spec[\'public_file_allowlist\'])\n    for rec in manifest[\'files\']:\n        path=root/rec[\'relative_path\'];assert path.resolve().is_relative_to(root.resolve())\n        assert path.stat().st_size==rec[\'bytes\'] and sha(path)==rec[\'sha256\']\n    provenance=json.loads((root/\'provenance.json\').read_text())\n    assert provenance[\'source_notebook_id\']==139418845 and provenance[\'source_script_version_id\']==359697707\n    assert provenance[\'source_worker_sha256\']==spec[\'training_worker_sha256\']\n    assert provenance[\'base_revision\']==spec[\'base_revision\'] and provenance[\'native_final_RMSNorm_retained\']\n    assert provenance[\'retained_layers\']==24 and provenance[\'pool_layers\']==[4,8,12,16,20,24] and provenance[\'feature_width\']==24576\n    assert provenance[\'successful_updates\']==450 and provenance[\'epochs\']==3 and provenance[\'final_not_OOF\']\n    assert provenance[\'final_overflow_attempts\']==0 and provenance[\'outer0_bounded_AMP_recoveries\']==4\n    assert provenance[\'classes\']==spec[\'classes\'] and provenance[\'synthetic_texts\']==spec[\'synthetic_texts\']\n    assert provenance[\'original_private_anchor_replay_passed\'] and provenance[\'actual_GPU_diagnostic_passed\']\n    config=json.loads((root/\'adapter/adapter_config.json\').read_text())\n    assert config[\'r\']==8 and config[\'lora_alpha\']==16 and config[\'lora_dropout\']==.05 and config[\'bias\']==\'none\'\n    assert set(config[\'target_modules\'])=={\'q_proj\',\'k_proj\',\'v_proj\',\'o_proj\',\'gate_proj\',\'up_proj\',\'down_proj\'}\n    assert config[\'auto_mapping\'][\'base_model_class\']==\'Olmo2Model\' and config[\'peft_type\']==\'LORA\'\n    for rec in spec[\'base_files\']:\n        path=base/rec[\'name\'];assert path.stat().st_size==rec[\'bytes\'] and sha(path)==rec[\'sha256\']\n    return root,base,provenance\nclass OLMoFinalPlain:\n    def __init__(self, spec_path, artifact_root, base_root, anchor_train_csv, diagnostic_only=False):\n        self.spec = json.loads(Path(spec_path).read_text())\n        for name, digest in self.spec["source_hashes"].items():\n            assert sha(Path(__file__).parent / name) == digest\n        checkpoint, base_dir, report = validate_contract(self.spec, artifact_root, base_root, diagnostic_only=diagnostic_only)\n        assert os.name != "nt" and Path("/kaggle/input").is_dir(), "Remote-only component"\n        os.environ["HF_HUB_OFFLINE"] = "1"\n        os.environ["TRANSFORMERS_OFFLINE"] = "1"\n        os.environ["TOKENIZERS_PARALLELISM"] = "false"\n        import importlib.util\n        finder = importlib.util.find_spec\n        importlib.util.find_spec = lambda name, *a, **kw: None if name.split(".")[0] == "torchao" else finder(name, *a, **kw)\n        # Keep the successful FULL worker\'s torchao mask active through PEFT dispatch.\n        import torch\n        from torch import nn\n        from peft import PeftModel\n        import peft.import_utils as peft_imports\n        peft_imports.is_torchao_available = lambda *a, **kw: False\n        import numpy as np\n        import pandas as pd\n        from safetensors.torch import load_file\n        from scipy.special import log_softmax, softmax\n        from olmo_owned_path import load_owned_base, make_representation, tokenize_owned\n\n        assert torch.cuda.is_available() and torch.cuda.get_device_capability(0) == (7, 5)\n        assert {name: importlib.metadata.version(name) for name in self.spec["core_versions"]} == self.spec["core_versions"]\n        torch.set_num_threads(2)\n        loading = []\n        # Fail before LM allocation on actual serialized readout geometry/class mismatch.\n        with np.load(checkpoint/\'readouts.npz\',allow_pickle=False) as checked:\n            np.testing.assert_array_equal(checked[\'classes\'],np.asarray(self.spec[\'classes\'],dtype=str))\n            np.testing.assert_array_equal(checked[\'pool_layers\'],[4,8,12,16,20,24])\n            for route in (\'plain\',\'center\'):\n                assert checked[route+\'_coef\'].shape==(18,24576) and checked[route+\'_coef\'].dtype==np.float32\n                assert checked[route+\'_intercept\'].shape==(18,) and checked[route+\'_intercept\'].dtype==np.float32\n        base = load_owned_base(base_dir, loading, lambda: None)\n        # Semantic resolution of actual serialized7suffixes; never assert serialized168 paths.\n        native_targets={name:module for name,module in base.named_modules() if isinstance(module,nn.Linear) and name.startswith(\'layers.\') and name.rsplit(\'.\',1)[-1] in {\'q_proj\',\'k_proj\',\'v_proj\',\'o_proj\',\'gate_proj\',\'up_proj\',\'down_proj\'}}\n        assert len(native_targets)==168\n        shapes={\'self_attn.q_proj\':(4096,4096),\'self_attn.k_proj\':(4096,4096),\'self_attn.v_proj\':(4096,4096),\'self_attn.o_proj\':(4096,4096),\'mlp.gate_proj\':(4096,11008),\'mlp.up_proj\':(4096,11008),\'mlp.down_proj\':(11008,4096)}\n        for layer in range(24):\n            for suffix,shape in shapes.items():\n                module=native_targets[f\'layers.{layer}.{suffix}\'];assert (module.in_features,module.out_features)==shape and module.bias is None\n        del native_targets,module\n        self.loading = loading\n        self.model = PeftModel.from_pretrained(base, checkpoint / "adapter", is_trainable=False, local_files_only=True).eval()\n        expanded=[name for name,module in self.model.named_modules() if hasattr(module,\'lora_A\')]\n        assert len(expanded)==168\n        assert len(self.model.base_model.model.layers)==24 and self.model.base_model.model.norm is not None\n        assert not any(parameter.requires_grad for parameter in self.model.parameters())\n        assert all(parameter.dtype == torch.float32 for name, parameter in self.model.named_parameters() if "lora_" in name)\n        self.layers = [4, 8, 12, 16, 20, 24]\n        self.norms = nn.ModuleList([nn.LayerNorm(4096) for _ in self.layers]).cuda().float()\n        saved = load_file(checkpoint / "heads.safetensors")\n        assert all(value.dtype == torch.float32 and bool(torch.isfinite(value).all()) for value in saved.values())\n        self.norms.load_state_dict({key.removeprefix("norms."): value for key, value in saved.items() if key.startswith("norms.")}, strict=True)\n        self.norms.eval()\n        self.norms.requires_grad_(False)\n        with np.load(checkpoint / "readouts.npz", allow_pickle=False) as readouts:\n            self.readouts = {key: readouts[key].copy() for key in readouts.files}\n        np.testing.assert_array_equal(self.readouts["classes"], np.asarray(self.spec["classes"], dtype=str))\n        np.testing.assert_array_equal(self.readouts["pool_layers"], self.layers)\n        assert self.readouts["plain_coef"].shape == self.readouts["center_coef"].shape == (18, 24576)\n        assert self.readouts["plain_intercept"].shape == self.readouts["center_intercept"].shape == (18,)\n        for route, prefix in (("plain", "plain"), ("center", "centered")):\n            for key, state_key in (("coef", "weight"), ("intercept", "bias")):\n                array = self.readouts[route + "_" + key]\n                assert array.dtype == np.float32 and np.isfinite(array).all()\n                np.testing.assert_array_equal(array, saved[prefix + "." + state_key].numpy())\n        # Synthetic literals only; canonical private B2 replay was checked before packaging.\n        self._diagnostic_only=False\n        self._torch,self._np=torch,np\n        self._make_representation,self._tokenize_owned=make_representation,tokenize_owned\n        self._log_softmax=log_softmax\n        with np.load(checkpoint/\'synthetic_anchors.npz\',allow_pickle=False) as anchors:\n            assert set(anchors.files)=={\'ids\',\'texts\',\'features\',\'plain_logp\',\'classes\'}\n            np.testing.assert_array_equal(anchors[\'ids\'],[\'synthetic-a\',\'synthetic-b\',\'synthetic-c\'])\n            np.testing.assert_array_equal(anchors[\'classes\'],self.readouts[\'classes\'])\n            texts=anchors[\'texts\'].tolist();assert texts==self.spec[\'synthetic_texts\']\n            expected=anchors[\'features\'].copy()\n            expected_logp=anchors[\'plain_logp\'].copy()\n            assert expected.shape==(3,24576) and expected.dtype==np.float32\n            assert expected_logp.shape==(3,18) and expected_logp.dtype==np.float32\n        self.tokenizer,_=tokenize_owned(base_dir,texts)\n        recorded_reloaded=expected\n        fresh = self._features(texts, sort_by_length=True)\n        fresh_logp=log_softmax(fresh @ self.readouts[\'plain_coef\'].T+self.readouts[\'plain_intercept\'],axis=1)\n        np.testing.assert_allclose(fresh_logp,expected_logp,rtol=0,atol=1e-6)\n        assert expected.dtype == fresh.dtype == np.float32 and np.isfinite(expected).all()\n        np.testing.assert_allclose(fresh, expected, rtol=1e-5, atol=1e-5)\n        np.testing.assert_allclose(fresh, recorded_reloaded, rtol=1e-5, atol=1e-5)\n        parity = {}\n        for route in ("plain", "center"):\n            a = expected if route == "plain" else expected - expected.mean(0)\n            b = fresh if route == "plain" else fresh - fresh.mean(0)\n            coef, intercept = self.readouts[route + "_coef"], self.readouts[route + "_intercept"]\n            before = softmax(log_softmax(a @ coef.T + intercept, axis=1), axis=1)\n            after = softmax(log_softmax(b @ coef.T + intercept, axis=1), axis=1)\n            parity[route] = float(np.max(np.abs(before - after)))\n            assert parity[route] <= 1e-6\n        self.anchor_replay = {"feature_maxabs": float(np.max(np.abs(fresh - expected))), "probability_maxabs": parity,\n                              "feature_rtol": 1e-5, "feature_atol": 1e-5, "probability_atol": 1e-6}\n\n    def _features(self, texts, sort_by_length=True):\n        torch, np = self._torch, self._np\n        tokenizer, tokens = self._tokenize_owned(None, texts, tokenizer=self.tokenizer)\n        representation = self._make_representation(self.model, tokenizer, tokens, self.layers, self.norms)\n        features = np.empty((len(tokens), 24576), dtype=np.float32)\n        order = np.argsort(np.asarray([len(row) for row in tokens]), kind="stable") if sort_by_length else np.arange(len(tokens))\n        with torch.inference_mode():\n            for start in range(0, len(order), 2):\n                positions = order[start:start + 2]\n                features[positions] = representation(positions)[0].cpu().numpy()\n        assert np.isfinite(features).all()\n        return features\n\n    def predict_plain_logp(self, ids, texts):\n        np = self._np\n        ids = np.asarray(ids, dtype=str)\n        assert ids.ndim == 1 and len(ids) == len(texts) > 0 and len(np.unique(ids)) == len(ids)\n        if self._diagnostic_only:\n            assert len(ids) in (3, 5) and ids.tolist() == self._diagnostic_ids[:len(ids)]\n            assert list(texts) == self._diagnostic_texts[:len(ids)], "Only fixed synthetic3/5-row diagnostic fixtures admitted"\n        features = self._features(texts)\n        logp = self._log_softmax(features @ self.readouts["plain_coef"].T + self.readouts["plain_intercept"], axis=1)\n        assert logp.shape == (len(ids), 18) and logp.dtype == np.float32 and np.isfinite(logp).all()\n        return {"ids": ids, "classes": self.readouts["classes"].copy(), "plain_logp": logp,\n                "features": features, "pool_layers": np.asarray(self.layers)}\n', 'olmo_owned_path.py': '"""Exact native final450 Olmo2Model prefix24 loader, tokenizer and representation."""\ndef load_owned_base(base_dir, loading, deadline):\n    import json,torch\n    from transformers import Olmo2Config,Olmo2Model\n    original = Olmo2Config.from_pretrained(base_dir, local_files_only=True, trust_remote_code=False)\n    assert (original.model_type, original.hidden_size, original.num_hidden_layers) == (\'olmo2\',4096,32)\n    assert (original.num_attention_heads, original.num_key_value_heads, original.vocab_size) == (32,32,100352)\n    assert original.intermediate_size == 11008 and not original.tie_word_embeddings and not original.attention_bias\n    assert original.bos_token_id is None and original.eos_token_id == 100257 and original.pad_token_id == 100277\n    assert original.rope_parameters[\'rope_theta\'] == 500000 and original.rope_parameters[\'rope_type\'] == \'default\'\n    assert original.rms_norm_eps == 1e-6 and original.attention_dropout == 0 and original.max_position_embeddings == 4096\n    assert not getattr(original,\'quantization_config\',None) and not getattr(original,\'auto_map\',None)\n    # Instantiate only the retained prefix: no full32 GPU model is ever constructed and pruned.\n    retained_config = Olmo2Config.from_dict(original.to_dict())\n    retained_config.num_hidden_layers = 24\n    assert retained_config.hidden_size == 4096 and retained_config.num_hidden_layers == 24\n    index = json.loads((base_dir/\'model.safetensors.index.json\').read_text())[\'weight_map\']\n    assert len(index) == 355 and \'lm_head.weight\' in index\n    original_layer_keys = {key for key in index if key.startswith(\'model.layers.\')}\n    assert len(original_layer_keys) == 352\n    assert {int(key.split(\'.\')[2]) for key in original_layer_keys} == set(range(32))\n    ignored_index_keys = {\'lm_head.weight\'} | {key for key in original_layer_keys if int(key.split(\'.\')[2]) >= 24}\n    used_index_keys = set(index) - ignored_index_keys\n    assert len(ignored_index_keys) == 89 and len(used_index_keys) == 266\n    expected_ignored = {key.removeprefix(\'model.\') for key in ignored_index_keys}\n    expected_used = {key.removeprefix(\'model.\') for key in used_index_keys}\n    def load_base():\n        deadline()\n        base, info = Olmo2Model.from_pretrained(base_dir, config=retained_config, dtype=torch.float16, device_map={"": 0},\n            attn_implementation="eager", trust_remote_code=False, local_files_only=True, output_loading_info=True)\n        assert not info.get("missing_keys") and not info.get("mismatched_keys") and not info.get("error_msgs"), info\n        assert {key.removeprefix(\'model.\') for key in info.get(\'unexpected_keys\', [])} == expected_ignored, info\n        assert set(base.state_dict()) == expected_used\n        assert sum(parameter.numel() for parameter in base.parameters()) == 5268443136\n        assert len(base.layers) == 24 and not hasattr(base, "lm_head")\n        assert base.__class__.__name__ == \'Olmo2Model\' and base.__class__.__module__.startswith(\'transformers.models.olmo2.\')\n        assert base.config._attn_implementation == \'eager\'\n        assert not any(name.endswith(\'lm_head\') for name, module in base.named_modules())\n        assert all(parameter.device.type == "cuda" and parameter.device.index == 0 for parameter in base.parameters())\n        assert all(parameter.dtype == torch.float16 for parameter in base.parameters())\n        base.requires_grad_(False)\n        loading.append(info)\n        return base\n    return load_base()\n\ndef make_representation(model,tokenizer,tokens,layers,norms):\n    import torch\n    def representation(rows):\n        batch = tokenizer.pad({"input_ids": [tokens[i] for i in rows]}, return_tensors="pt").to("cuda:0")\n        with torch.autocast("cuda", dtype=torch.float16):\n            hidden = model(**batch, output_hidden_states=True, use_cache=False).hidden_states\n        assert len(hidden) == 25\n        pieces = []\n        with torch.autocast("cuda", enabled=False):\n            for layer, norm in zip(layers, norms):\n                mask = batch.attention_mask.unsqueeze(-1).float()\n                mask[:, 0] = 0\n                assert bool((mask.sum(1) > 0).all())\n                mean = (hidden[layer].float() * mask).sum(1) / mask.sum(1)\n                pieces.append(norm(mean))\n            z = torch.cat(pieces, -1)\n        assert bool(torch.isfinite(z).all())\n        return z, int(batch.attention_mask.sum()), int(batch.input_ids.numel())\n    return representation\n\ndef tokenize_owned(base_dir,texts,tokenizer=None):\n    from transformers import AutoTokenizer,Olmo2Config\n    if tokenizer is None:\n        original = Olmo2Config.from_pretrained(base_dir,local_files_only=True,trust_remote_code=False)\n        tokenizer = AutoTokenizer.from_pretrained(str(base_dir), use_fast=True, local_files_only=True, trust_remote_code=False)\n        assert tokenizer.is_fast and tokenizer.__class__.__module__.startswith(\'transformers.\')\n        # Official tokenizer metadata aliases BOS to EOS; model config has no native BOS.\n        assert tokenizer.bos_token_id == tokenizer.eos_token_id == 100257 and tokenizer.pad_token_id == 100277\n        assert len(tokenizer) == 100278 and original.vocab_size == 100352\n        assert tokenizer.convert_tokens_to_ids(\'<|endoftext|>\') == 100257\n        assert tokenizer.convert_tokens_to_ids(\'<|pad|>\') == 100277\n        tokenizer.padding_side = \'right\'\n    content = tokenizer(texts, add_special_tokens=False, truncation=True, max_length=446)[\'input_ids\']\n    tokens = [[100257]+row+[100257] for row in content]\n    assert all(2 <= len(row) <= 448 and row[0] == row[-1] == 100257 for row in tokens)\n    del content\n    return tokenizer,tokens', 'olmo-spec.json': '{\n  "diagnostic_only_authorized": false,\n  "training_worker_sha256": "b15a93fa754261c1b8e0176684bdc04b1b4692a5c63ee040394b9e93fa8e882d",\n  "inference_authorized": true,\n  "source_hashes": {\n    "olmo_component.py": "32c7f5c4f89bced4e21bc5e852e1f9086b95d10842f009d5799a5c10be875f45",\n    "olmo_owned_path.py": "8e702264a084b93be8dbce9b1c50e59cd9b12ca0ded11ff67a7da73bbc642649"\n  },\n  "core_versions": {\n    "torch": "2.11.0+cu128",\n    "transformers": "5.16.1",\n    "peft": "0.20.0",\n    "accelerate": "1.14.0",\n    "safetensors": "0.8.0",\n    "numpy": "2.1.3",\n    "pandas": "2.3.3"\n  },\n  "base_files": [\n    {\n      "name": "model-00001-of-00006.safetensors",\n      "bytes": 2474775744,\n      "sha256": "215fd4ad7d0f5d83a6bbb2322cc2c76080b1fdac05e438606210442220c4ed9f"\n    },\n    {\n      "name": "model-00002-of-00006.safetensors",\n      "bytes": 2495815296,\n      "sha256": "08575313e1863e7c7af7ff12831033342ad39deea4635545eaf27b65d4518e51"\n    },\n    {\n      "name": "model-00003-of-00006.safetensors",\n      "bytes": 2462277424,\n      "sha256": "8b575321f45f7b7347358c75fb36d10bb3b3422802327c97c66eedea2718ec9d"\n    },\n    {\n      "name": "model-00004-of-00006.safetensors",\n      "bytes": 2428706272,\n      "sha256": "b46c6e7a37da4a188bbff066a4bea020e2576c49fb6c735e6208b33de0f0e76a"\n    },\n    {\n      "name": "model-00005-of-00006.safetensors",\n      "bytes": 2428706272,\n      "sha256": "1b89fd1f9c7d8712a48471bfaf31a26be0e83932ebe3b7d111fffd9aa0560362"\n    },\n    {\n      "name": "model-00006-of-00006.safetensors",\n      "bytes": 2306994712,\n      "sha256": "d131e899a6209c67b8c046bf747ae4e03850179c79f4406e73ced08671c748ff"\n    },\n    {\n      "name": "config.json",\n      "bytes": 623,\n      "sha256": "f1e210de0ba704c579768c154bf57e36e170868ffecb060a6e0d3f2e6b0b9710"\n    },\n    {\n      "name": "conversion-progress.json",\n      "bytes": 88783,\n      "sha256": "5d5856493d980f5cc2371c6e16769cb627174fc5cf09c8419353b4413a5240b9"\n    },\n    {\n      "name": "generation_config.json",\n      "bytes": 126,\n      "sha256": "73e7f06e4ea172c6edeb1a44cab4a603e76962a85ea7ef3245769ba46ca23a30"\n    },\n    {\n      "name": "LICENSE_APACHE_2_0.txt",\n      "bytes": 11358,\n      "sha256": "cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30"\n    },\n    {\n      "name": "merges.txt",\n      "bytes": 916646,\n      "sha256": "b6fe424e334903f7fb84d3a106d9730455f4744b9fe3c21ee136d97a00e72502"\n    },\n    {\n      "name": "MODEL_WEIGHT_LICENSE_DECLARATION.txt",\n      "bytes": 603,\n      "sha256": "8ff0e78dea979a376700398e2e63467a5f6c39d53079f710897055489ba9faa7"\n    },\n    {\n      "name": "model.safetensors.index.json",\n      "bytes": 29593,\n      "sha256": "76424611ebaa0c3320284c7fb7a8fe8a05355f7ce4d0688097ae40ba634e02a3"\n    },\n    {\n      "name": "NOTICE.md",\n      "bytes": 690,\n      "sha256": "46c3931560a02beb073d6bc7b2d39d03330e289712ab1d8eb94c1e7a0c844201"\n    },\n    {\n      "name": "README.md",\n      "bytes": 12098,\n      "sha256": "a87d5b59422c6040caf0fc62cbcb5f7ace6c71489c1173cf6d9e000777eac962"\n    },\n    {\n      "name": "special_tokens_map.json",\n      "bytes": 125,\n      "sha256": "3c6bf7c09d5473c303cee8575a22bb51e5153c17d177a721b43cd4785c6d09ae"\n    },\n    {\n      "name": "tokenizer_config.json",\n      "bytes": 4345,\n      "sha256": "91c69c665697785ace4ec7dbd159e1839bc9bb5033ab05a56bb0547521dc9ab0"\n    },\n    {\n      "name": "tokenizer.json",\n      "bytes": 7137177,\n      "sha256": "73fd5254624f39a88e3faac6a8e11300fc3c735ed37880d4f4f08db898eaecca"\n    },\n    {\n      "name": "upstream-model.safetensors.index.json",\n      "bytes": 29594,\n      "sha256": "c285ec59423a5a7203724c6258f61d691c0816e4c3f79b1fede776298f894b07"\n    },\n    {\n      "name": "vocab.json",\n      "bytes": 1611056,\n      "sha256": "9e14712c91b37c7aab74b1306baa46ac342d620637a4b44523cdc3aec7d24195"\n    }\n  ],\n  "synthetic_ids": [\n    "synthetic-a",\n    "synthetic-b",\n    "synthetic-c",\n    "synthetic-d",\n    "synthetic-e"\n  ],\n  "classes": [\n    "claude-fable-5",\n    "claude-opus-4-6",\n    "claude-opus-4-8",\n    "claude-opus-5",\n    "claude-sonnet-5",\n    "gemini-3.1-pro-preview",\n    "gemini-3.6-flash",\n    "gemini-3.7-flash",\n    "gemini-3.8-flash",\n    "glm-5.3",\n    "glm-5.3-flash",\n    "gpt-5.5",\n    "gpt-5.6-luna",\n    "gpt-5.6-sol",\n    "gpt-5.6-terra",\n    "gpt-6-astra",\n    "kimi-k3",\n    "qwen3.8-max"\n  ],\n  "base_revision": "7df9a82518afdecae4e8c026b27adccc8c1f0032",\n  "production_admitted": true,\n  "synthetic_texts": [\n    "The quiet observatory opened before sunrise.",\n    "A small program sorts seven coloured stones.",\n    "Please compare the two imaginary calendars carefully."\n  ],\n  "train_sha256": "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707",\n  "base_manifest_sha256": "09fbb318fb7d036162159cc8936e7bd18ce6c24fa8ab192cbfdedaf390045fbe",\n  "diagnostic_cap_seconds": 1200,\n  "no_auto_retry": true,\n  "root_production_admitted": true,\n  "public_asset_release_audited": true,\n  "compact_manifest_sha256": "3ccc36f4a36bc21e09c43a8a32b0d7f6216004cf872835dd41e55925aab51f85",\n  "actual_GPU_diagnostic_audit_sha256": "7e3f981139841f5686648e20bdf365b9da7b5906c1692e8921f9aa813da443bf",\n  "public_file_allowlist": [\n    "adapter/adapter_model.safetensors",\n    "adapter/adapter_config.json",\n    "heads.safetensors",\n    "readouts.npz",\n    "synthetic_anchors.npz",\n    "LICENSE_APACHE_2_0.txt",\n    "NOTICE.md",\n    "provenance.json"\n  ],\n  "changed_public_loader_runtime_audit_sha256": "f7b8168579a8d5bf2ac564a7f86033a176326f90b282192cda9903425a1dfa7a",\n  "base_ref": "lightningv08/olmo2-7b-fp16-offline-weights/1",\n  "public_dataset_ref": "lightningv08/ntl-olmo2-native24-final-inference-assets/2",\n  "public_dataset_id": 12692007\n}\n', 'worker_barrier.py': '"""File barrier after native loading/replay and before each owned query forward."""\nimport json,os,time\nfrom pathlib import Path\n\ndef ready_and_wait(name,timeout=900):\n    assert name in (\'olmo\',\'smol\')\n    root=Path(os.environ[\'NTL_CANDIDATE_TEMP\']).resolve()\n    assert root.is_relative_to(Path(\'/kaggle/temp\').resolve())\n    assert os.environ[\'CUDA_VISIBLE_DEVICES\'].startswith(\'GPU-\')\n    path=root/(name+\'-READY.json\');assert not path.exists()\n    path.write_text(json.dumps(dict(name=name,PID=os.getpid(),UUID=os.environ[\'CUDA_VISIBLE_DEVICES\'])))\n    started=time.monotonic()\n    while not (root/\'OWNED-GO\').exists():\n        assert time.monotonic()-started<timeout,\'Bounded READY/GO wait; no retry\'\n        time.sleep(.1)\n', 'smol_owned_path.py': '"""Minimal byte-preserved scientific functions extracted from the reviewed owned full-fit path."""\n\n\ndef load_owned_base(base_dir, loading, deadline):\n    import torch\n    from transformers import AutoModel\n    def load_base():\n        deadline()\n        base, info = AutoModel.from_pretrained(base_dir, dtype=torch.float16, device_map={"": 0},\n            attn_implementation="sdpa", trust_remote_code=False, local_files_only=True, output_loading_info=True)\n        assert not info.get("missing_keys") and not info.get("mismatched_keys") and not info.get("error_msgs"), info\n        assert set(info.get("unexpected_keys", [])) <= {"lm_head.weight"}, info\n        assert len(base.layers) == 36 and not hasattr(base, "lm_head")\n        assert all(parameter.device.type == "cuda" and parameter.device.index == 0 for parameter in base.parameters())\n        base.requires_grad_(False)\n        loading.append(info)\n        return base\n    return load_base()\n\n\ndef make_representation(model, tokenizer, tokens, layers, norms):\n    import torch\n    def representation(rows):\n        batch = tokenizer.pad({"input_ids": [tokens[i] for i in rows]}, return_tensors="pt").to("cuda:0")\n        with torch.autocast("cuda", dtype=torch.float16):\n            hidden = model(**batch, output_hidden_states=True, use_cache=False).hidden_states\n        assert len(hidden) == 37\n        pieces = []\n        with torch.autocast("cuda", enabled=False):\n            for layer, norm in zip(layers, norms):\n                mask = batch.attention_mask.unsqueeze(-1).float()\n                mask[:, 0] = 0\n                assert bool((mask.sum(1) > 0).all())\n                mean = (hidden[layer].float() * mask).sum(1) / mask.sum(1)\n                pieces.append(norm(mean))\n            z = torch.cat(pieces, -1)\n        assert bool(torch.isfinite(z).all())\n        return z, int(batch.attention_mask.sum()), int(batch.input_ids.numel())\n    return representation\n\n\ndef tokenize_owned(base_dir, texts, tokenizer=None):\n    if tokenizer is None:\n        from transformers import AutoTokenizer\n        tokenizer = AutoTokenizer.from_pretrained(base_dir, local_files_only=True, trust_remote_code=False)\n    assert tokenizer.bos_token_id is None and tokenizer.eos_token_id == 128001\n    tokenizer.padding_side = "right"\n    tokenizer.pad_token_id = 128004\n    tokens = tokenizer(texts, add_special_tokens=False, truncation=True, max_length=447)["input_ids"]\n    tokens = [row + [128001] for row in tokens]\n    assert all(2 <= len(row) <= 448 for row in tokens)\n    return tokenizer, tokens', 'smol_readout.py': '"""Native NumPy FP32 routing plus a frozen FP64 Gaussian; no fitting."""\nimport numpy as np\nfrom center_readout import source_logsoftmax,add_logprob\nfrom gaussian_core import scores\n\ndef smol_score(Z,ro,groups,eligible,prm,gaussian_fn=scores):\n    assert Z.dtype==np.float32 and Z.ndim==2 and Z.shape[1]==12288 and np.isfinite(Z).all()\n    n=len(Z);groups=np.asarray(groups);eligible=np.asarray(eligible)\n    assert groups.shape==eligible.shape==(n,) and groups.dtype.kind in \'iu\' and eligible.dtype==bool\n    counts=np.unique(groups,return_inverse=True,return_counts=True)\n    sizes=counts[2][counts[1]]\n    np.testing.assert_array_equal(eligible,(sizes>=2)&(sizes<=18))\n    for route in (\'plain\',\'center\'):\n        assert ro[route+\'_coef\'].dtype==ro[route+\'_intercept\'].dtype==np.float32\n        assert ro[route+\'_coef\'].shape==(18,12288) and ro[route+\'_intercept\'].shape==(18,)\n    # Exactly the validated CPU plain-head order; clipping belongs ONLY here.\n    lg=Z@ro[\'plain_coef\'].T+ro[\'plain_intercept\']\n    lg-=lg.max(1,keepdims=True)\n    p=np.exp(lg)\n    plain=add_logprob(p/p.sum(1,keepdims=True))\n    routed=plain.copy()\n    local=np.flatnonzero(eligible)\n    if len(local):\n        # One query pool; all members of every eligible original group are present.\n        X=np.asarray(Z[local],dtype=np.float32).copy()\n        local_groups=groups[local]\n        for group in np.unique(local_groups):\n            i=local_groups==group\n            X[i]-=X[i].mean(0)\n        lc=X@ro[\'center_coef\'].T+ro[\'center_intercept\']\n        routed[local]=source_logsoftmax(lc)\n    assert routed.dtype==np.float32 and np.isfinite(routed).all()\n    result=routed.astype(np.float64)\n    if len(local):\n        group_labels=np.unique(groups[local],return_inverse=True)[1]\n        gaussian=gaussian_fn(Z[local],group_labels,prm)\n        assert gaussian.dtype==np.float64 and gaussian.shape==(len(local),18) and np.isfinite(gaussian).all()\n        np.testing.assert_allclose(np.exp(gaussian).sum(1),1,rtol=0,atol=1e-10)\n        result[local]+=0.10*gaussian\n    assert result.dtype==np.float64 and np.isfinite(result).all()\n    # No member renormalization, exponentiation or clipping after Gaussian addition.\n    return result,routed\n', 'center_readout.py': 'import numpy as np\n\ndef source_logsoftmax(value):\n    value = value - value.max(1, keepdims=True)\n    return value - np.log(np.exp(value).sum(1, keepdims=True))\n\n\ndef add_logprob(p):\n    assert np.isfinite(p).all() and (p >= 0).all()\n    np.testing.assert_allclose(p.sum(-1), 1, rtol=0, atol=2e-5)\n    return np.log(np.clip(p, 1e-9, 1))\n\n\ndef lora_readouts(Z, ro, pools, recovered, adaptive):\n    """Production FP32 affine and pool-local query centering, never global centering."""\n    assert Z.dtype == np.float32 and Z.shape == (539, 12288)\n    assert ro[\'plain_coef\'].dtype == ro[\'center_coef\'].dtype == np.float32\n    lg = Z @ ro[\'plain_coef\'].T + ro[\'plain_intercept\']\n    lg -= lg.max(1, keepdims=True)\n    p = np.exp(lg)\n    plain = add_logprob(p / p.sum(1, keepdims=True))\n    centered = np.zeros((3, 539, 18), dtype=np.float32)\n    for context in range(3):\n        for pool in np.unique(pools[context]):\n            local = np.flatnonzero((pools[context] == pool) & adaptive[context])\n            if not len(local):\n                continue\n            X = np.asarray(Z[local], dtype=np.float32).copy()\n            groups = recovered[context, local]\n            for group in np.unique(groups):\n                i = groups == group\n                X[i] -= X[i].mean(0)\n            lc = X @ ro[\'center_coef\'].T + ro[\'center_intercept\']\n            centered[context, local] = source_logsoftmax(lc)\n    LC = None\n    if \'claude_coef\' in ro:\n        LC = source_logsoftmax(Z @ ro[\'claude_coef\'].T + ro[\'claude_intercept\'])\n    return plain, centered, LC\n\n\n', 'gaussian_core.py': '"""Shared-prompt Gaussian core by prvsiyan, Apache-2.0.\nExtracted without mathematical changes from The Prompt Is Not the Author,\nv1/script359792979: https://www.kaggle.com/code/prvsiyan/the-prompt-is-not-the-author\n"""\nimport numpy as np\nimport itertools\nK=18\nDEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,\n           "iters": 5, "repel": True, "sp_scale": 1.0}\n\n_MASK = {}\n\nLAM_DIFF = 1e6\n\ndef _lse(A, axis=None, keepdims=False):\n    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)\n    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m\n    if keepdims: return out\n    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())\n\ndef log_softmax(L): return L - _lse(L, axis=1, keepdims=True)\n\ndef _lw_shrinkage(X):\n    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""\n    from sklearn.covariance import ledoit_wolf_shrinkage\n    return float(ledoit_wolf_shrinkage(X, assume_centered=True))\n\ndef _shrunk(S, a):\n    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)\n\ndef _two_way(Z, y, pinv, G, iters=4):\n    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""\n    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)\n    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))\n    for _ in range(iters):\n        R = Z - P[pinv]\n        for c in cls: mu[c] = R[y == c].mean(0)\n        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]\n    return mu, P, cnt_p\n\ndef fit(X, y, prompts, cfg=None):\n    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.\n    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller\'s cross-fit)."""\n    c = dict(DEFAULT); c.update(cfg or {})\n    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape\n    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)\n    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx\n    if c["red"] == "none":\n        V = np.eye(D)\n    else:\n        if c["red"] == "wpca":\n            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]\n        elif c["red"] == "pca": R = Xs\n        else: raise ValueError(c["red"])\n        d = int(min(c["d"], N - G - 1, D))\n        # economy SVD through the (N x N) Gram matrix (N << D for the real views)\n        if N < D:\n            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]\n            V = (R.T @ U) / np.sqrt(w)\n        else:\n            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T\n    Z = Xs @ V; d = Z.shape[1]\n    mu, P, ng = _two_way(Z, y, pinv, G)\n    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)\n    Se_raw = E.T @ E / dof\n    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)\n    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw\n    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])\n    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]\n    # joint diagonalisation: A Se A\' = I, A Sp A\' = diag(lam)\n    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)\n    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li\n    W = V @ A.T\n    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,\n            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}\n\ndef transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]\n\ndef _inj_mask(n, C):\n    if (n, C) not in _MASK:\n        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)\n        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]\n        _MASK[(n, C)] = bad\n    return _MASK[(n, C)]\n\ndef singleton_ll(U, m, lam):\n    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""\n    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)\n\ndef joint_terms(U, m, lam):\n    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""\n    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]\n    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)\n    return a, B\n\ndef joint_table(a, B):\n    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""\n    n, C = a.shape; Q = np.zeros([C] * n)\n    for i in range(n):\n        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)\n    for i, j in itertools.combinations(range(n), 2):\n        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)\n    return Q\n\ndef exact_marginals(Q, T=1.0, inj=True):\n    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""\n    n = Q.ndim; C = Q.shape[0]; Qt = Q / T\n    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)\n    lz = _lse(Qt); out = np.empty((n, C))\n    for i in range(n):\n        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz\n    return out\n\ndef em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):\n    """cavity EM for one group (any n >= 2): returns (n, C) log q."""\n    n = len(U); v = lam / (1 + (n - 1) * lam)\n    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))\n    q = np.exp(log_softmax(L))\n    for _ in range(iters):\n        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)\n        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T\n        if repel:   # mean-field at-most-once: discourage classes the other members already hold\n            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)\n        q = np.exp(log_softmax(L))\n    return log_softmax(L)\n\ndef lam_g(prm, cfg=None):\n    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg \'diff\'): then the joint likelihood\n    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs\n    centred prototypes) and the prompt term cancels exactly."""\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})\n    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]\n\ndef group_logpost(U, prm, T1=None, Tg=None, cfg=None):\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)\n    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg\n    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)\n    if n <= c["n_exact"]:\n        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])\n    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])\n\ndef scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):\n    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""\n    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))\n    sizes = np.bincount(lab); s1 = sizes[lab] == 1\n    T1 = prm["T1"] if T1 is None else T1\n    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)\n    for g in np.where(sizes >= 2)[0]:\n        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)\n    return out\n', 'volume-spec.json': '{\n  "manifest_sha256": "c634333fe86aaefa093bce6b6813c048cb41269e429db47b701913ffb940ed2a",\n  "base_files": [\n    {\n      "name": "README.md",\n      "bytes": 12340,\n      "sha256": "9bb1fff003df8f9497d2589c345b80817af6864a39938c14edc5207bc5717009"\n    },\n    {\n      "name": "config.json",\n      "bytes": 943,\n      "sha256": "e8336e843aecb733691a043ad4209168e1730158b5e03c2672591cc946036a47"\n    },\n    {\n      "name": "generation_config.json",\n      "bytes": 126,\n      "sha256": "7d830c1a274c4484a50eb34ed68cdbd3611a4981fd4d539c7ec1fed67e01cec4"\n    },\n    {\n      "name": "model-00001-of-00002.safetensors",\n      "bytes": 4966315264,\n      "sha256": "7e270ac568ee1880ddbadad66ccdcd9906d52415e8904e2f300c75250b9c7d49"\n    },\n    {\n      "name": "model-00002-of-00002.safetensors",\n      "bytes": 1183919744,\n      "sha256": "c6a6e7690a66dcc386a6a9b456686e7c308d45cba884d116c928dafa7fa987ae"\n    },\n    {\n      "name": "model.safetensors.index.json",\n      "bytes": 26899,\n      "sha256": "e3a7254c086c4a78f95ad5864f435d165ffed1ed8998326ce5107b4423c15b76"\n    },\n    {\n      "name": "special_tokens_map.json",\n      "bytes": 151,\n      "sha256": "9faaa15579bbf433176493a621142ff580ae2d13d81bf4702cb72f4d6902dc62"\n    },\n    {\n      "name": "tokenizer.json",\n      "bytes": 17208861,\n      "sha256": "ab4da6b2aa68247e9c0fa9b97fc7fcc796505038d01f7e144522a65ce0dbd2e5"\n    },\n    {\n      "name": "tokenizer_config.json",\n      "bytes": 50405,\n      "sha256": "f398d9edb0184aac1c9fd1b1bdcd3a5a1527ae560fd847719635a66e2b54e9cd"\n    }\n  ],\n  "core_versions": {\n    "torch": "2.11.0+cu128",\n    "transformers": "5.16.1",\n    "peft": "0.20.0",\n    "accelerate": "1.14.0",\n    "safetensors": "0.8.0",\n    "numpy": "2.1.3",\n    "pandas": "2.3.3"\n  }\n}', 'diagnostic-assets.json': '{"copied_files": {"adapter/README.md": "external-volume-training/rank1/checkpoint/adapter/README.md", "adapter/adapter_config.json": "external-volume-training/rank1/checkpoint/adapter/adapter_config.json", "adapter/adapter_model.safetensors": "external-volume-training/rank1/checkpoint/adapter/adapter_model.safetensors", "heads.safetensors": "external-volume-training/rank1/checkpoint/heads.safetensors", "readouts.npz": "external-volume-training/rank1/checkpoint/readouts.npz", "gaussian-final.npz": "volume-evaluation/gaussian-final.npz", "synthetic-anchors.npz": "volume-evaluation/synthetic-anchors.npz", "synthetic3.json": "volume-evaluation/synthetic3.json"}, "binary_literal_files": {"README.md": "IyBTbW9sTE0zIEZ1bGwgRXh0ZXJuYWwtVm9sdW1lIEluZmVyZW5jZQoKQSBmdWxsIDM2LWxheWVyIFNtb2xMTTMtM0IgY2xhc3NpZmllciB0cmFpbmVkIGZpcnN0IG9uIDI4LDM5MCBkaXN0aW5jdCBlbGlnaWJsZSBleHRlcm5hbCByZXNwb25zZXMsIHRoZW4gb24gYWxsIDIsNjk5IG9mZmljaWFsIHRyYWluaW5nIHJlc3BvbnNlcy4gU2l4IG1hc2tlZC1tZWFuIHBvb2xzIHVzZSB0cmFpbmVkIExheWVyTm9ybXMsIHdpdGggc2VwYXJhdGUgcGxhaW4gYW5kIHByb21wdC1jZW50ZXJlZCBoZWFkcy4KClRoZSBleHRlcm5hbCBzdGFnZSBjb3ZlcnMgMiwyMTcgcHJvbXB0cyBpbiAxLDY5NiB1cGRhdGVzLiBTYW1lLXByb21wdCB0cmlwbGV0cyBjb250YWluIGRpc3RpbmN0IGF1dGhvciBsYWJlbHM7IGZpeGVkIGludmVyc2UgcHJlc2VudGF0aW9uLWZyZXF1ZW5jeSB3ZWlnaHRzIGJhbGFuY2UgdGhlIGNsYXNzIG9iamVjdGl2ZS4gVGhlIG9mZmljaWFsIGFkYXB0YXRpb24gdXNlcyA0NTAgdXBkYXRlcy4gVGhlc2UgZmluYWwgd2VpZ2h0cyBhbmQgcmVhZG91dHMgdXNlIHRoZSBjb21wbGV0ZSBvZmZpY2lhbCB0cmFpbmluZyBzZXQgYW5kIGFyZSAqKm5vdCBvdXQtb2YtZm9sZCoqLgoKT24gYSBzZXBhcmF0ZSBvZmZpY2lhbCBwcm9tcHQtaGVsZC1vdXQgZGV2ZWxvcG1lbnQgc3BsaXQsIHRoZSBsYXJnZXIgZXh0ZXJuYWwgc3RhZ2UgaW1wcm92ZWQgdGhlIG1lYW4gbmF0aXZlLWhlYWQgQmFsYW5jZWQgQWNjdXJhY3kgYnkgMC4wNTg3NyByZWxhdGl2ZSB0byB0aGUgZWFybGllciA1LDQwMC1yZXNwb25zZSB3YXJtdXAuIFRoZSBwYWlyZWQgMzAtcHJvbXB0IGludGVydmFsIHdhcyBbMC4wMzU5MywgMC4wODIxMF0uIFRoaXMgcmV1c2VkIGRldmVsb3BtZW50IGNvbXBhcmlzb24gaXMgY29uZGl0aW9uYWwgb24gZXh0ZXJuYWwgcHJvbXB0IGluZGVwZW5kZW5jZSBhbmQgZG9lcyBub3QgZXN0YWJsaXNoIGEgbGVhZGVyYm9hcmQgZ2FpbiBvciBmdWxsLWVuc2VtYmxlIE9PRiBzY29yZS4KCmBzeW50aGV0aWMzLmpzb25gIGNvbnRhaW5zIHRocmVlIGludmVudGVkIHBhc3NhZ2VzIGZvciByZXByb2R1Y2liaWxpdHkgY2hlY2tzLiBUaGUgY29ycmVzcG9uZGluZyBudW1lcmljYWwgYW5jaG9ycyB3ZXJlIGdlbmVyYXRlZCBvbiBhIEthZ2dsZSBUNC4gTm8gcmF3IGNvbXBldGl0aW9uIHRleHRzLCB0cmFpbmluZyBmZWF0dXJlIGNhY2hlLCBvcHRpbWl6ZXIgc3RhdGUgb3IgaW50ZXJtZWRpYXRlIGNoZWNrcG9pbnQgaXMgaW5jbHVkZWQuCgojIyBTb3VyY2VzIGFuZCBsaWNlbnNlcwoKLSBTbW9sTE0zLTNCLUJhc2U6IFtIdWdnaW5nRmFjZVRCXShodHRwczovL2h1Z2dpbmdmYWNlLmNvL0h1Z2dpbmdGYWNlVEIvU21vbExNMy0zQi1CYXNlKSwgcmV2aXNpb24gYGQ3OGE0MmY3OTE5ODYwM2U2MTQwOTU3NTM0ODRhMDRjMTBjMmI5NDBgLCBBcGFjaGUgMi4wLgotIEV4dGVybmFsIHBvb2w6IFtOaXNoYW50IERhaGFsJ3MgTlRMIFR3by1TdGFnZSBMb1JBIGRhdGFzZXRdKGh0dHBzOi8vd3d3LmthZ2dsZS5jb20vZGF0YXNldHMvbmlzaGFudGRhaGFsL250bC10d28tc3RhZ2UtbG9yYSksIHZlcnNpb24gMSwgQ0MgQlktU0EgNC4wLiBSZXRhaW4gaXRzIHVwc3RyZWFtIGNvbnRyaWJ1dG9yIGF0dHJpYnV0aW9uIGFuZCBzaGFyZS1hbGlrZSB0ZXJtczogW2xpY2Vuc2VdKGh0dHBzOi8vY3JlYXRpdmVjb21tb25zLm9yZy9saWNlbnNlcy9ieS1zYS80LjAvKS4KLSBTaGFyZWQtcHJvbXB0IEdhdXNzaWFuIG1ldGhvZDogW3BydnNpeWFuXShodHRwczovL3d3dy5rYWdnbGUuY29tL2NvZGUvcHJ2c2l5YW4vdGhlLXByb21wdC1pcy1ub3QtdGhlLWF1dGhvciksIEFwYWNoZSAyLjA7IHRoZSBvcHRpb25hbCBwcmUtZml0dGVkIEdhdXNzaWFuIHVzZXMgb2ZmaWNpYWwgdHJhaW5pbmcgZmVhdHVyZXMgb25seS4KClRoZSBzb3VyY2UvbW9kZWwgQXBhY2hlIG5vdGljZXMgYW5kIGV4dGVybmFsLXBvb2wgQ0MgQlktU0EgdGVybXMgcmV0YWluIHRoZWlyIHJlc3BlY3RpdmUgc2NvcGUuIFRoZSBLYWdnbGUgY29sbGVjdGlvbidzIE90aGVyIGxpY2Vuc2UgY2F0ZWdvcnkgZG9lcyBub3QgcmVwbGFjZSB0aGVzZSB0ZXJtcy4K", "LICENSE_APACHE_2_0.txt": "CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIEFwYWNoZSBMaWNlbnNlCiAgICAgICAgICAgICAgICAgICAgICAgICAgIFZlcnNpb24gMi4wLCBKYW51YXJ5IDIwMDQKICAgICAgICAgICAgICAgICAgICAgICAgaHR0cDovL3d3dy5hcGFjaGUub3JnL2xpY2Vuc2VzLwoKICAgVEVSTVMgQU5EIENPTkRJVElPTlMgRk9SIFVTRSwgUkVQUk9EVUNUSU9OLCBBTkQgRElTVFJJQlVUSU9OCgogICAxLiBEZWZpbml0aW9ucy4KCiAgICAgICJMaWNlbnNlIiBzaGFsbCBtZWFuIHRoZSB0ZXJtcyBhbmQgY29uZGl0aW9ucyBmb3IgdXNlLCByZXByb2R1Y3Rpb24sCiAgICAgIGFuZCBkaXN0cmlidXRpb24gYXMgZGVmaW5lZCBieSBTZWN0aW9ucyAxIHRocm91Z2ggOSBvZiB0aGlzIGRvY3VtZW50LgoKICAgICAgIkxpY2Vuc29yIiBzaGFsbCBtZWFuIHRoZSBjb3B5cmlnaHQgb3duZXIgb3IgZW50aXR5IGF1dGhvcml6ZWQgYnkKICAgICAgdGhlIGNvcHlyaWdodCBvd25lciB0aGF0IGlzIGdyYW50aW5nIHRoZSBMaWNlbnNlLgoKICAgICAgIkxlZ2FsIEVudGl0eSIgc2hhbGwgbWVhbiB0aGUgdW5pb24gb2YgdGhlIGFjdGluZyBlbnRpdHkgYW5kIGFsbAogICAgICBvdGhlciBlbnRpdGllcyB0aGF0IGNvbnRyb2wsIGFyZSBjb250cm9sbGVkIGJ5LCBvciBhcmUgdW5kZXIgY29tbW9uCiAgICAgIGNvbnRyb2wgd2l0aCB0aGF0IGVudGl0eS4gRm9yIHRoZSBwdXJwb3NlcyBvZiB0aGlzIGRlZmluaXRpb24sCiAgICAgICJjb250cm9sIiBtZWFucyAoaSkgdGhlIHBvd2VyLCBkaXJlY3Qgb3IgaW5kaXJlY3QsIHRvIGNhdXNlIHRoZQogICAgICBkaXJlY3Rpb24gb3IgbWFuYWdlbWVudCBvZiBzdWNoIGVudGl0eSwgd2hldGhlciBieSBjb250cmFjdCBvcgogICAgICBvdGhlcndpc2UsIG9yIChpaSkgb3duZXJzaGlwIG9mIGZpZnR5IHBlcmNlbnQgKDUwJSkgb3IgbW9yZSBvZiB0aGUKICAgICAgb3V0c3RhbmRpbmcgc2hhcmVzLCBvciAoaWlpKSBiZW5lZmljaWFsIG93bmVyc2hpcCBvZiBzdWNoIGVudGl0eS4KCiAgICAgICJZb3UiIChvciAiWW91ciIpIHNoYWxsIG1lYW4gYW4gaW5kaXZpZHVhbCBvciBMZWdhbCBFbnRpdHkKICAgICAgZXhlcmNpc2luZyBwZXJtaXNzaW9ucyBncmFudGVkIGJ5IHRoaXMgTGljZW5zZS4KCiAgICAgICJTb3VyY2UiIGZvcm0gc2hhbGwgbWVhbiB0aGUgcHJlZmVycmVkIGZvcm0gZm9yIG1ha2luZyBtb2RpZmljYXRpb25zLAogICAgICBpbmNsdWRpbmcgYnV0IG5vdCBsaW1pdGVkIHRvIHNvZnR3YXJlIHNvdXJjZSBjb2RlLCBkb2N1bWVudGF0aW9uCiAgICAgIHNvdXJjZSwgYW5kIGNvbmZpZ3VyYXRpb24gZmlsZXMuCgogICAgICAiT2JqZWN0IiBmb3JtIHNoYWxsIG1lYW4gYW55IGZvcm0gcmVzdWx0aW5nIGZyb20gbWVjaGFuaWNhbAogICAgICB0cmFuc2Zvcm1hdGlvbiBvciB0cmFuc2xhdGlvbiBvZiBhIFNvdXJjZSBmb3JtLCBpbmNsdWRpbmcgYnV0CiAgICAgIG5vdCBsaW1pdGVkIHRvIGNvbXBpbGVkIG9iamVjdCBjb2RlLCBnZW5lcmF0ZWQgZG9jdW1lbnRhdGlvbiwKICAgICAgYW5kIGNvbnZlcnNpb25zIHRvIG90aGVyIG1lZGlhIHR5cGVzLgoKICAgICAgIldvcmsiIHNoYWxsIG1lYW4gdGhlIHdvcmsgb2YgYXV0aG9yc2hpcCwgd2hldGhlciBpbiBTb3VyY2Ugb3IKICAgICAgT2JqZWN0IGZvcm0sIG1hZGUgYXZhaWxhYmxlIHVuZGVyIHRoZSBMaWNlbnNlLCBhcyBpbmRpY2F0ZWQgYnkgYQogICAgICBjb3B5cmlnaHQgbm90aWNlIHRoYXQgaXMgaW5jbHVkZWQgaW4gb3IgYXR0YWNoZWQgdG8gdGhlIHdvcmsKICAgICAgKGFuIGV4YW1wbGUgaXMgcHJvdmlkZWQgaW4gdGhlIEFwcGVuZGl4IGJlbG93KS4KCiAgICAgICJEZXJpdmF0aXZlIFdvcmtzIiBzaGFsbCBtZWFuIGFueSB3b3JrLCB3aGV0aGVyIGluIFNvdXJjZSBvciBPYmplY3QKICAgICAgZm9ybSwgdGhhdCBpcyBiYXNlZCBvbiAob3IgZGVyaXZlZCBmcm9tKSB0aGUgV29yayBhbmQgZm9yIHdoaWNoIHRoZQogICAgICBlZGl0b3JpYWwgcmV2aXNpb25zLCBhbm5vdGF0aW9ucywgZWxhYm9yYXRpb25zLCBvciBvdGhlciBtb2RpZmljYXRpb25zCiAgICAgIHJlcHJlc2VudCwgYXMgYSB3aG9sZSwgYW4gb3JpZ2luYWwgd29yayBvZiBhdXRob3JzaGlwLiBGb3IgdGhlIHB1cnBvc2VzCiAgICAgIG9mIHRoaXMgTGljZW5zZSwgRGVyaXZhdGl2ZSBXb3JrcyBzaGFsbCBub3QgaW5jbHVkZSB3b3JrcyB0aGF0IHJlbWFpbgogICAgICBzZXBhcmFibGUgZnJvbSwgb3IgbWVyZWx5IGxpbmsgKG9yIGJpbmQgYnkgbmFtZSkgdG8gdGhlIGludGVyZmFjZXMgb2YsCiAgICAgIHRoZSBXb3JrIGFuZCBEZXJpdmF0aXZlIFdvcmtzIHRoZXJlb2YuCgogICAgICAiQ29udHJpYnV0aW9uIiBzaGFsbCBtZWFuIGFueSB3b3JrIG9mIGF1dGhvcnNoaXAsIGluY2x1ZGluZwogICAgICB0aGUgb3JpZ2luYWwgdmVyc2lvbiBvZiB0aGUgV29yayBhbmQgYW55IG1vZGlmaWNhdGlvbnMgb3IgYWRkaXRpb25zCiAgICAgIHRvIHRoYXQgV29yayBvciBEZXJpdmF0aXZlIFdvcmtzIHRoZXJlb2YsIHRoYXQgaXMgaW50ZW50aW9uYWxseQogICAgICBzdWJtaXR0ZWQgdG8gTGljZW5zb3IgZm9yIGluY2x1c2lvbiBpbiB0aGUgV29yayBieSB0aGUgY29weXJpZ2h0IG93bmVyCiAgICAgIG9yIGJ5IGFuIGluZGl2aWR1YWwgb3IgTGVnYWwgRW50aXR5IGF1dGhvcml6ZWQgdG8gc3VibWl0IG9uIGJlaGFsZiBvZgogICAgICB0aGUgY29weXJpZ2h0IG93bmVyLiBGb3IgdGhlIHB1cnBvc2VzIG9mIHRoaXMgZGVmaW5pdGlvbiwgInN1Ym1pdHRlZCIKICAgICAgbWVhbnMgYW55IGZvcm0gb2YgZWxlY3Ryb25pYywgdmVyYmFsLCBvciB3cml0dGVuIGNvbW11bmljYXRpb24gc2VudAogICAgICB0byB0aGUgTGljZW5zb3Igb3IgaXRzIHJlcHJlc2VudGF0aXZlcywgaW5jbHVkaW5nIGJ1dCBub3QgbGltaXRlZCB0bwogICAgICBjb21tdW5pY2F0aW9uIG9uIGVsZWN0cm9uaWMgbWFpbGluZyBsaXN0cywgc291cmNlIGNvZGUgY29udHJvbCBzeXN0ZW1zLAogICAgICBhbmQgaXNzdWUgdHJhY2tpbmcgc3lzdGVtcyB0aGF0IGFyZSBtYW5hZ2VkIGJ5LCBvciBvbiBiZWhhbGYgb2YsIHRoZQogICAgICBMaWNlbnNvciBmb3IgdGhlIHB1cnBvc2Ugb2YgZGlzY3Vzc2luZyBhbmQgaW1wcm92aW5nIHRoZSBXb3JrLCBidXQKICAgICAgZXhjbHVkaW5nIGNvbW11bmljYXRpb24gdGhhdCBpcyBjb25zcGljdW91c2x5IG1hcmtlZCBvciBvdGhlcndpc2UKICAgICAgZGVzaWduYXRlZCBpbiB3cml0aW5nIGJ5IHRoZSBjb3B5cmlnaHQgb3duZXIgYXMgIk5vdCBhIENvbnRyaWJ1dGlvbi4iCgogICAgICAiQ29udHJpYnV0b3IiIHNoYWxsIG1lYW4gTGljZW5zb3IgYW5kIGFueSBpbmRpdmlkdWFsIG9yIExlZ2FsIEVudGl0eQogICAgICBvbiBiZWhhbGYgb2Ygd2hvbSBhIENvbnRyaWJ1dGlvbiBoYXMgYmVlbiByZWNlaXZlZCBieSBMaWNlbnNvciBhbmQKICAgICAgc3Vic2VxdWVudGx5IGluY29ycG9yYXRlZCB3aXRoaW4gdGhlIFdvcmsuCgogICAyLiBHcmFudCBvZiBDb3B5cmlnaHQgTGljZW5zZS4gU3ViamVjdCB0byB0aGUgdGVybXMgYW5kIGNvbmRpdGlvbnMgb2YKICAgICAgdGhpcyBMaWNlbnNlLCBlYWNoIENvbnRyaWJ1dG9yIGhlcmVieSBncmFudHMgdG8gWW91IGEgcGVycGV0dWFsLAogICAgICB3b3JsZHdpZGUsIG5vbi1leGNsdXNpdmUsIG5vLWNoYXJnZSwgcm95YWx0eS1mcmVlLCBpcnJldm9jYWJsZQogICAgICBjb3B5cmlnaHQgbGljZW5zZSB0byByZXByb2R1Y2UsIHByZXBhcmUgRGVyaXZhdGl2ZSBXb3JrcyBvZiwKICAgICAgcHVibGljbHkgZGlzcGxheSwgcHVibGljbHkgcGVyZm9ybSwgc3VibGljZW5zZSwgYW5kIGRpc3RyaWJ1dGUgdGhlCiAgICAgIFdvcmsgYW5kIHN1Y2ggRGVyaXZhdGl2ZSBXb3JrcyBpbiBTb3VyY2Ugb3IgT2JqZWN0IGZvcm0uCgogICAzLiBHcmFudCBvZiBQYXRlbnQgTGljZW5zZS4gU3ViamVjdCB0byB0aGUgdGVybXMgYW5kIGNvbmRpdGlvbnMgb2YKICAgICAgdGhpcyBMaWNlbnNlLCBlYWNoIENvbnRyaWJ1dG9yIGhlcmVieSBncmFudHMgdG8gWW91IGEgcGVycGV0dWFsLAogICAgICB3b3JsZHdpZGUsIG5vbi1leGNsdXNpdmUsIG5vLWNoYXJnZSwgcm95YWx0eS1mcmVlLCBpcnJldm9jYWJsZQogICAgICAoZXhjZXB0IGFzIHN0YXRlZCBpbiB0aGlzIHNlY3Rpb24pIHBhdGVudCBsaWNlbnNlIHRvIG1ha2UsIGhhdmUgbWFkZSwKICAgICAgdXNlLCBvZmZlciB0byBzZWxsLCBzZWxsLCBpbXBvcnQsIGFuZCBvdGhlcndpc2UgdHJhbnNmZXIgdGhlIFdvcmssCiAgICAgIHdoZXJlIHN1Y2ggbGljZW5zZSBhcHBsaWVzIG9ubHkgdG8gdGhvc2UgcGF0ZW50IGNsYWltcyBsaWNlbnNhYmxlCiAgICAgIGJ5IHN1Y2ggQ29udHJpYnV0b3IgdGhhdCBhcmUgbmVjZXNzYXJpbHkgaW5mcmluZ2VkIGJ5IHRoZWlyCiAgICAgIENvbnRyaWJ1dGlvbihzKSBhbG9uZSBvciBieSBjb21iaW5hdGlvbiBvZiB0aGVpciBDb250cmlidXRpb24ocykKICAgICAgd2l0aCB0aGUgV29yayB0byB3aGljaCBzdWNoIENvbnRyaWJ1dGlvbihzKSB3YXMgc3VibWl0dGVkLiBJZiBZb3UKICAgICAgaW5zdGl0dXRlIHBhdGVudCBsaXRpZ2F0aW9uIGFnYWluc3QgYW55IGVudGl0eSAoaW5jbHVkaW5nIGEKICAgICAgY3Jvc3MtY2xhaW0gb3IgY291bnRlcmNsYWltIGluIGEgbGF3c3VpdCkgYWxsZWdpbmcgdGhhdCB0aGUgV29yawogICAgICBvciBhIENvbnRyaWJ1dGlvbiBpbmNvcnBvcmF0ZWQgd2l0aGluIHRoZSBXb3JrIGNvbnN0aXR1dGVzIGRpcmVjdAogICAgICBvciBjb250cmlidXRvcnkgcGF0ZW50IGluZnJpbmdlbWVudCwgdGhlbiBhbnkgcGF0ZW50IGxpY2Vuc2VzCiAgICAgIGdyYW50ZWQgdG8gWW91IHVuZGVyIHRoaXMgTGljZW5zZSBmb3IgdGhhdCBXb3JrIHNoYWxsIHRlcm1pbmF0ZQogICAgICBhcyBvZiB0aGUgZGF0ZSBzdWNoIGxpdGlnYXRpb24gaXMgZmlsZWQuCgogICA0LiBSZWRpc3RyaWJ1dGlvbi4gWW91IG1heSByZXByb2R1Y2UgYW5kIGRpc3RyaWJ1dGUgY29waWVzIG9mIHRoZQogICAgICBXb3JrIG9yIERlcml2YXRpdmUgV29ya3MgdGhlcmVvZiBpbiBhbnkgbWVkaXVtLCB3aXRoIG9yIHdpdGhvdXQKICAgICAgbW9kaWZpY2F0aW9ucywgYW5kIGluIFNvdXJjZSBvciBPYmplY3QgZm9ybSwgcHJvdmlkZWQgdGhhdCBZb3UKICAgICAgbWVldCB0aGUgZm9sbG93aW5nIGNvbmRpdGlvbnM6CgogICAgICAoYSkgWW91IG11c3QgZ2l2ZSBhbnkgb3RoZXIgcmVjaXBpZW50cyBvZiB0aGUgV29yayBvcgogICAgICAgICAgRGVyaXZhdGl2ZSBXb3JrcyBhIGNvcHkgb2YgdGhpcyBMaWNlbnNlOyBhbmQKCiAgICAgIChiKSBZb3UgbXVzdCBjYXVzZSBhbnkgbW9kaWZpZWQgZmlsZXMgdG8gY2FycnkgcHJvbWluZW50IG5vdGljZXMKICAgICAgICAgIHN0YXRpbmcgdGhhdCBZb3UgY2hhbmdlZCB0aGUgZmlsZXM7IGFuZAoKICAgICAgKGMpIFlvdSBtdXN0IHJldGFpbiwgaW4gdGhlIFNvdXJjZSBmb3JtIG9mIGFueSBEZXJpdmF0aXZlIFdvcmtzCiAgICAgICAgICB0aGF0IFlvdSBkaXN0cmlidXRlLCBhbGwgY29weXJpZ2h0LCBwYXRlbnQsIHRyYWRlbWFyaywgYW5kCiAgICAgICAgICBhdHRyaWJ1dGlvbiBub3RpY2VzIGZyb20gdGhlIFNvdXJjZSBmb3JtIG9mIHRoZSBXb3JrLAogICAgICAgICAgZXhjbHVkaW5nIHRob3NlIG5vdGljZXMgdGhhdCBkbyBub3QgcGVydGFpbiB0byBhbnkgcGFydCBvZgogICAgICAgICAgdGhlIERlcml2YXRpdmUgV29ya3M7IGFuZAoKICAgICAgKGQpIElmIHRoZSBXb3JrIGluY2x1ZGVzIGEgIk5PVElDRSIgdGV4dCBmaWxlIGFzIHBhcnQgb2YgaXRzCiAgICAgICAgICBkaXN0cmlidXRpb24sIHRoZW4gYW55IERlcml2YXRpdmUgV29ya3MgdGhhdCBZb3UgZGlzdHJpYnV0ZSBtdXN0CiAgICAgICAgICBpbmNsdWRlIGEgcmVhZGFibGUgY29weSBvZiB0aGUgYXR0cmlidXRpb24gbm90aWNlcyBjb250YWluZWQKICAgICAgICAgIHdpdGhpbiBzdWNoIE5PVElDRSBmaWxlLCBleGNsdWRpbmcgdGhvc2Ugbm90aWNlcyB0aGF0IGRvIG5vdAogICAgICAgICAgcGVydGFpbiB0byBhbnkgcGFydCBvZiB0aGUgRGVyaXZhdGl2ZSBXb3JrcywgaW4gYXQgbGVhc3Qgb25lCiAgICAgICAgICBvZiB0aGUgZm9sbG93aW5nIHBsYWNlczogd2l0aGluIGEgTk9USUNFIHRleHQgZmlsZSBkaXN0cmlidXRlZAogICAgICAgICAgYXMgcGFydCBvZiB0aGUgRGVyaXZhdGl2ZSBXb3Jrczsgd2l0aGluIHRoZSBTb3VyY2UgZm9ybSBvcgogICAgICAgICAgZG9jdW1lbnRhdGlvbiwgaWYgcHJvdmlkZWQgYWxvbmcgd2l0aCB0aGUgRGVyaXZhdGl2ZSBXb3Jrczsgb3IsCiAgICAgICAgICB3aXRoaW4gYSBkaXNwbGF5IGdlbmVyYXRlZCBieSB0aGUgRGVyaXZhdGl2ZSBXb3JrcywgaWYgYW5kCiAgICAgICAgICB3aGVyZXZlciBzdWNoIHRoaXJkLXBhcnR5IG5vdGljZXMgbm9ybWFsbHkgYXBwZWFyLiBUaGUgY29udGVudHMKICAgICAgICAgIG9mIHRoZSBOT1RJQ0UgZmlsZSBhcmUgZm9yIGluZm9ybWF0aW9uYWwgcHVycG9zZXMgb25seSBhbmQKICAgICAgICAgIGRvIG5vdCBtb2RpZnkgdGhlIExpY2Vuc2UuIFlvdSBtYXkgYWRkIFlvdXIgb3duIGF0dHJpYnV0aW9uCiAgICAgICAgICBub3RpY2VzIHdpdGhpbiBEZXJpdmF0aXZlIFdvcmtzIHRoYXQgWW91IGRpc3RyaWJ1dGUsIGFsb25nc2lkZQogICAgICAgICAgb3IgYXMgYW4gYWRkZW5kdW0gdG8gdGhlIE5PVElDRSB0ZXh0IGZyb20gdGhlIFdvcmssIHByb3ZpZGVkCiAgICAgICAgICB0aGF0IHN1Y2ggYWRkaXRpb25hbCBhdHRyaWJ1dGlvbiBub3RpY2VzIGNhbm5vdCBiZSBjb25zdHJ1ZWQKICAgICAgICAgIGFzIG1vZGlmeWluZyB0aGUgTGljZW5zZS4KCiAgICAgIFlvdSBtYXkgYWRkIFlvdXIgb3duIGNvcHlyaWdodCBzdGF0ZW1lbnQgdG8gWW91ciBtb2RpZmljYXRpb25zIGFuZAogICAgICBtYXkgcHJvdmlkZSBhZGRpdGlvbmFsIG9yIGRpZmZlcmVudCBsaWNlbnNlIHRlcm1zIGFuZCBjb25kaXRpb25zCiAgICAgIGZvciB1c2UsIHJlcHJvZHVjdGlvbiwgb3IgZGlzdHJpYnV0aW9uIG9mIFlvdXIgbW9kaWZpY2F0aW9ucywgb3IKICAgICAgZm9yIGFueSBzdWNoIERlcml2YXRpdmUgV29ya3MgYXMgYSB3aG9sZSwgcHJvdmlkZWQgWW91ciB1c2UsCiAgICAgIHJlcHJvZHVjdGlvbiwgYW5kIGRpc3RyaWJ1dGlvbiBvZiB0aGUgV29yayBvdGhlcndpc2UgY29tcGxpZXMgd2l0aAogICAgICB0aGUgY29uZGl0aW9ucyBzdGF0ZWQgaW4gdGhpcyBMaWNlbnNlLgoKICAgNS4gU3VibWlzc2lvbiBvZiBDb250cmlidXRpb25zLiBVbmxlc3MgWW91IGV4cGxpY2l0bHkgc3RhdGUgb3RoZXJ3aXNlLAogICAgICBhbnkgQ29udHJpYnV0aW9uIGludGVudGlvbmFsbHkgc3VibWl0dGVkIGZvciBpbmNsdXNpb24gaW4gdGhlIFdvcmsKICAgICAgYnkgWW91IHRvIHRoZSBMaWNlbnNvciBzaGFsbCBiZSB1bmRlciB0aGUgdGVybXMgYW5kIGNvbmRpdGlvbnMgb2YKICAgICAgdGhpcyBMaWNlbnNlLCB3aXRob3V0IGFueSBhZGRpdGlvbmFsIHRlcm1zIG9yIGNvbmRpdGlvbnMuCiAgICAgIE5vdHdpdGhzdGFuZGluZyB0aGUgYWJvdmUsIG5vdGhpbmcgaGVyZWluIHNoYWxsIHN1cGVyc2VkZSBvciBtb2RpZnkKICAgICAgdGhlIHRlcm1zIG9mIGFueSBzZXBhcmF0ZSBsaWNlbnNlIGFncmVlbWVudCB5b3UgbWF5IGhhdmUgZXhlY3V0ZWQKICAgICAgd2l0aCBMaWNlbnNvciByZWdhcmRpbmcgc3VjaCBDb250cmlidXRpb25zLgoKICAgNi4gVHJhZGVtYXJrcy4gVGhpcyBMaWNlbnNlIGRvZXMgbm90IGdyYW50IHBlcm1pc3Npb24gdG8gdXNlIHRoZSB0cmFkZQogICAgICBuYW1lcywgdHJhZGVtYXJrcywgc2VydmljZSBtYXJrcywgb3IgcHJvZHVjdCBuYW1lcyBvZiB0aGUgTGljZW5zb3IsCiAgICAgIGV4Y2VwdCBhcyByZXF1aXJlZCBmb3IgcmVhc29uYWJsZSBhbmQgY3VzdG9tYXJ5IHVzZSBpbiBkZXNjcmliaW5nIHRoZQogICAgICBvcmlnaW4gb2YgdGhlIFdvcmsgYW5kIHJlcHJvZHVjaW5nIHRoZSBjb250ZW50IG9mIHRoZSBOT1RJQ0UgZmlsZS4KCiAgIDcuIERpc2NsYWltZXIgb2YgV2FycmFudHkuIFVubGVzcyByZXF1aXJlZCBieSBhcHBsaWNhYmxlIGxhdyBvcgogICAgICBhZ3JlZWQgdG8gaW4gd3JpdGluZywgTGljZW5zb3IgcHJvdmlkZXMgdGhlIFdvcmsgKGFuZCBlYWNoCiAgICAgIENvbnRyaWJ1dG9yIHByb3ZpZGVzIGl0cyBDb250cmlidXRpb25zKSBvbiBhbiAiQVMgSVMiIEJBU0lTLAogICAgICBXSVRIT1VUIFdBUlJBTlRJRVMgT1IgQ09ORElUSU9OUyBPRiBBTlkgS0lORCwgZWl0aGVyIGV4cHJlc3Mgb3IKICAgICAgaW1wbGllZCwgaW5jbHVkaW5nLCB3aXRob3V0IGxpbWl0YXRpb24sIGFueSB3YXJyYW50aWVzIG9yIGNvbmRpdGlvbnMKICAgICAgb2YgVElUTEUsIE5PTi1JTkZSSU5HRU1FTlQsIE1FUkNIQU5UQUJJTElUWSwgb3IgRklUTkVTUyBGT1IgQQogICAgICBQQVJUSUNVTEFSIFBVUlBPU0UuIFlvdSBhcmUgc29sZWx5IHJlc3BvbnNpYmxlIGZvciBkZXRlcm1pbmluZyB0aGUKICAgICAgYXBwcm9wcmlhdGVuZXNzIG9mIHVzaW5nIG9yIHJlZGlzdHJpYnV0aW5nIHRoZSBXb3JrIGFuZCBhc3N1bWUgYW55CiAgICAgIHJpc2tzIGFzc29jaWF0ZWQgd2l0aCBZb3VyIGV4ZXJjaXNlIG9mIHBlcm1pc3Npb25zIHVuZGVyIHRoaXMgTGljZW5zZS4KCiAgIDguIExpbWl0YXRpb24gb2YgTGlhYmlsaXR5LiBJbiBubyBldmVudCBhbmQgdW5kZXIgbm8gbGVnYWwgdGhlb3J5LAogICAgICB3aGV0aGVyIGluIHRvcnQgKGluY2x1ZGluZyBuZWdsaWdlbmNlKSwgY29udHJhY3QsIG9yIG90aGVyd2lzZSwKICAgICAgdW5sZXNzIHJlcXVpcmVkIGJ5IGFwcGxpY2FibGUgbGF3IChzdWNoIGFzIGRlbGliZXJhdGUgYW5kIGdyb3NzbHkKICAgICAgbmVnbGlnZW50IGFjdHMpIG9yIGFncmVlZCB0byBpbiB3cml0aW5nLCBzaGFsbCBhbnkgQ29udHJpYnV0b3IgYmUKICAgICAgbGlhYmxlIHRvIFlvdSBmb3IgZGFtYWdlcywgaW5jbHVkaW5nIGFueSBkaXJlY3QsIGluZGlyZWN0LCBzcGVjaWFsLAogICAgICBpbmNpZGVudGFsLCBvciBjb25zZXF1ZW50aWFsIGRhbWFnZXMgb2YgYW55IGNoYXJhY3RlciBhcmlzaW5nIGFzIGEKICAgICAgcmVzdWx0IG9mIHRoaXMgTGljZW5zZSBvciBvdXQgb2YgdGhlIHVzZSBvciBpbmFiaWxpdHkgdG8gdXNlIHRoZQogICAgICBXb3JrIChpbmNsdWRpbmcgYnV0IG5vdCBsaW1pdGVkIHRvIGRhbWFnZXMgZm9yIGxvc3Mgb2YgZ29vZHdpbGwsCiAgICAgIHdvcmsgc3RvcHBhZ2UsIGNvbXB1dGVyIGZhaWx1cmUgb3IgbWFsZnVuY3Rpb24sIG9yIGFueSBhbmQgYWxsCiAgICAgIG90aGVyIGNvbW1lcmNpYWwgZGFtYWdlcyBvciBsb3NzZXMpLCBldmVuIGlmIHN1Y2ggQ29udHJpYnV0b3IKICAgICAgaGFzIGJlZW4gYWR2aXNlZCBvZiB0aGUgcG9zc2liaWxpdHkgb2Ygc3VjaCBkYW1hZ2VzLgoKICAgOS4gQWNjZXB0aW5nIFdhcnJhbnR5IG9yIEFkZGl0aW9uYWwgTGlhYmlsaXR5LiBXaGlsZSByZWRpc3RyaWJ1dGluZwogICAgICB0aGUgV29yayBvciBEZXJpdmF0aXZlIFdvcmtzIHRoZXJlb2YsIFlvdSBtYXkgY2hvb3NlIHRvIG9mZmVyLAogICAgICBhbmQgY2hhcmdlIGEgZmVlIGZvciwgYWNjZXB0YW5jZSBvZiBzdXBwb3J0LCB3YXJyYW50eSwgaW5kZW1uaXR5LAogICAgICBvciBvdGhlciBsaWFiaWxpdHkgb2JsaWdhdGlvbnMgYW5kL29yIHJpZ2h0cyBjb25zaXN0ZW50IHdpdGggdGhpcwogICAgICBMaWNlbnNlLiBIb3dldmVyLCBpbiBhY2NlcHRpbmcgc3VjaCBvYmxpZ2F0aW9ucywgWW91IG1heSBhY3Qgb25seQogICAgICBvbiBZb3VyIG93biBiZWhhbGYgYW5kIG9uIFlvdXIgc29sZSByZXNwb25zaWJpbGl0eSwgbm90IG9uIGJlaGFsZgogICAgICBvZiBhbnkgb3RoZXIgQ29udHJpYnV0b3IsIGFuZCBvbmx5IGlmIFlvdSBhZ3JlZSB0byBpbmRlbW5pZnksCiAgICAgIGRlZmVuZCwgYW5kIGhvbGQgZWFjaCBDb250cmlidXRvciBoYXJtbGVzcyBmb3IgYW55IGxpYWJpbGl0eQogICAgICBpbmN1cnJlZCBieSwgb3IgY2xhaW1zIGFzc2VydGVkIGFnYWluc3QsIHN1Y2ggQ29udHJpYnV0b3IgYnkgcmVhc29uCiAgICAgIG9mIHlvdXIgYWNjZXB0aW5nIGFueSBzdWNoIHdhcnJhbnR5IG9yIGFkZGl0aW9uYWwgbGlhYmlsaXR5LgoKICAgRU5EIE9GIFRFUk1TIEFORCBDT05ESVRJT05TCgogICBBUFBFTkRJWDogSG93IHRvIGFwcGx5IHRoZSBBcGFjaGUgTGljZW5zZSB0byB5b3VyIHdvcmsuCgogICAgICBUbyBhcHBseSB0aGUgQXBhY2hlIExpY2Vuc2UgdG8geW91ciB3b3JrLCBhdHRhY2ggdGhlIGZvbGxvd2luZwogICAgICBib2lsZXJwbGF0ZSBub3RpY2UsIHdpdGggdGhlIGZpZWxkcyBlbmNsb3NlZCBieSBicmFja2V0cyAiW10iCiAgICAgIHJlcGxhY2VkIHdpdGggeW91ciBvd24gaWRlbnRpZnlpbmcgaW5mb3JtYXRpb24uIChEb24ndCBpbmNsdWRlCiAgICAgIHRoZSBicmFja2V0cyEpICBUaGUgdGV4dCBzaG91bGQgYmUgZW5jbG9zZWQgaW4gdGhlIGFwcHJvcHJpYXRlCiAgICAgIGNvbW1lbnQgc3ludGF4IGZvciB0aGUgZmlsZSBmb3JtYXQuIFdlIGFsc28gcmVjb21tZW5kIHRoYXQgYQogICAgICBmaWxlIG9yIGNsYXNzIG5hbWUgYW5kIGRlc2NyaXB0aW9uIG9mIHB1cnBvc2UgYmUgaW5jbHVkZWQgb24gdGhlCiAgICAgIHNhbWUgInByaW50ZWQgcGFnZSIgYXMgdGhlIGNvcHlyaWdodCBub3RpY2UgZm9yIGVhc2llcgogICAgICBpZGVudGlmaWNhdGlvbiB3aXRoaW4gdGhpcmQtcGFydHkgYXJjaGl2ZXMuCgogICBDb3B5cmlnaHQgW3l5eXldIFtuYW1lIG9mIGNvcHlyaWdodCBvd25lcl0KCiAgIExpY2Vuc2VkIHVuZGVyIHRoZSBBcGFjaGUgTGljZW5zZSwgVmVyc2lvbiAyLjAgKHRoZSAiTGljZW5zZSIpOwogICB5b3UgbWF5IG5vdCB1c2UgdGhpcyBmaWxlIGV4Y2VwdCBpbiBjb21wbGlhbmNlIHdpdGggdGhlIExpY2Vuc2UuCiAgIFlvdSBtYXkgb2J0YWluIGEgY29weSBvZiB0aGUgTGljZW5zZSBhdAoKICAgICAgIGh0dHA6Ly93d3cuYXBhY2hlLm9yZy9saWNlbnNlcy9MSUNFTlNFLTIuMAoKICAgVW5sZXNzIHJlcXVpcmVkIGJ5IGFwcGxpY2FibGUgbGF3IG9yIGFncmVlZCB0byBpbiB3cml0aW5nLCBzb2Z0d2FyZQogICBkaXN0cmlidXRlZCB1bmRlciB0aGUgTGljZW5zZSBpcyBkaXN0cmlidXRlZCBvbiBhbiAiQVMgSVMiIEJBU0lTLAogICBXSVRIT1VUIFdBUlJBTlRJRVMgT1IgQ09ORElUSU9OUyBPRiBBTlkgS0lORCwgZWl0aGVyIGV4cHJlc3Mgb3IgaW1wbGllZC4KICAgU2VlIHRoZSBMaWNlbnNlIGZvciB0aGUgc3BlY2lmaWMgbGFuZ3VhZ2UgZ292ZXJuaW5nIHBlcm1pc3Npb25zIGFuZAogICBsaW1pdGF0aW9ucyB1bmRlciB0aGUgTGljZW5zZS4K", "volume-inference-manifest.json": "ew0KICAib2ZmaWNpYWxfdXBkYXRlcyI6IDQ1MCwNCiAgInRyYWluaW5nX3ZlcnNpb24iOiAxLA0KICAidHJhaW5pbmdfc2NyaXB0X3ZlcnNpb24iOiAzNjAxNzkwMTUsDQogICJyYXdfY29tcGV0aXRpb25fdGV4dHMiOiBmYWxzZSwNCiAgInRyYWluaW5nX3NvdXJjZV9zaGEyNTYiOiAiOTQ2MGVkYjk0MDkxYzA1YzVkNzFjMzE2NjNmMWNkZGJmZWNlNjYwMjJkNDk1M2Y1OWU4NzRjMjljMmRmNTVkZCIsDQogICJpc19PT0YiOiBmYWxzZSwNCiAgIm9wdGlvbmFsX2dhdXNzaWFuIjogIlByZS1maXR0ZWQgb2ZmaWNpYWwtb25seTsgbm90IHNlbGVjdGVkIGZvciB0aGUgY3VycmVudCBjYW5kaWRhdGUiLA0KICAiYmFzZV9yZXZpc2lvbiI6ICJkNzhhNDJmNzkxOTg2MDNlNjE0MDk1NzUzNDg0YTA0YzEwYzJiOTQwIiwNCiAgIm9mZmljaWFsX3Jvd3MiOiAyNjk5LA0KICAib3B0aW1pemVyX3N0YXRlIjogZmFsc2UsDQogICJ0cmFpbmluZ19ub3RlYm9vayI6ICJsaWdodG5pbmd2MDgvbnRsLXNtb2wtZnVsbC1leHRlcm5hbC10cmFpbmluZyIsDQogICJmaWxlcyI6IHsNCiAgICAiYWRhcHRlci9hZGFwdGVyX2NvbmZpZy5qc29uIjogew0KICAgICAgInNoYTI1NiI6ICJhOGYyMzk4OGNkMjYwZWZkZDg1Yzc5OWQ1ODEwMDZiMWMzZDQ4NjFjYmE3ZWNmZGI5M2NhN2NiNGFhNTFhYTY4IiwNCiAgICAgICJieXRlcyI6IDEzMTcNCiAgICB9LA0KICAgICJMSUNFTlNFX0FQQUNIRV8yXzAudHh0Ijogew0KICAgICAgInNoYTI1NiI6ICJjZmM3NzQ5Yjk2ZjYzYmQzMWMzYzQyYjVjNDcxYmY3NTY4MTQwNTNlODQ3YzEwZjNlYjAwMzQxN2JjNTIzZDMwIiwNCiAgICAgICJieXRlcyI6IDExMzU4DQogICAgfSwNCiAgICAiaGVhZHMuc2FmZXRlbnNvcnMiOiB7DQogICAgICAic2hhMjU2IjogIjE4MWI2OWU2NTZmOWEyMWQ1ZDJkMjBiYjkxMWZiNjMwZjBiY2I0OTFjZDM0ZGExM2IwMGM3ZmZjM2M4ZTg1NzEiLA0KICAgICAgImJ5dGVzIjogMTg2OTE2OA0KICAgIH0sDQogICAgInJlYWRvdXRzLm5weiI6IHsNCiAgICAgICJzaGEyNTYiOiAiZjNmMTNkNjhmMWRjMDZjNzliMjE1ZTlmYzdjYjg0YzI2NWMxODQ2NmZmMDQxYTEwZTc0YzUwNzY5OGZjMmZkNCIsDQogICAgICAiYnl0ZXMiOiAxNjQzNTM0DQogICAgfSwNCiAgICAiYWRhcHRlci9hZGFwdGVyX21vZGVsLnNhZmV0ZW5zb3JzIjogew0KICAgICAgInNoYTI1NiI6ICI5ZWQ5ODQ0NzRkZTVlMTBmMjEyYWJkZjhkMTk4Zjk5OTEwNWEzOTIwN2E0MzQ0YjU3ZGNlYzhmNWEyNmE0ZDZiIiwNCiAgICAgICJieXRlcyI6IDEyMDk3ODE3Ng0KICAgIH0sDQogICAgIlJFQURNRS5tZCI6IHsNCiAgICAgICJzaGEyNTYiOiAiNGUyYjM0NDY1NmY4OTlhMGVjOTNkOGFmNzU2ZDIwYWM4ODA4M2VjOWE1NjQwYTQ1YjI4ZmRmYWE0OTRjMDNkYSIsDQogICAgICAiYnl0ZXMiOiAyMDkxDQogICAgfSwNCiAgICAiYWRhcHRlci9SRUFETUUubWQiOiB7DQogICAgICAic2hhMjU2IjogIjRmYmI4ZDc0MTVmNjdkMDY1ODY2OWFmOWJkODgzYmVjZmNmOGZmYzFmZTExN2IyYTMwZWQ4MzA3OTViNWNjNzIiLA0KICAgICAgImJ5dGVzIjogNTI4Ng0KICAgIH0sDQogICAgInN5bnRoZXRpYzMuanNvbiI6IHsNCiAgICAgICJzaGEyNTYiOiAiZDMwYzJmMWFlYzc2NzdkMDUzMWQyY2E1MzBiMzdkYzE5MDYwYmQ5ZmU5MzczNDAxOTRhOTY4OWFlYTNmZDA3NiIsDQogICAgICAiYnl0ZXMiOiAyOTYNCiAgICB9LA0KICAgICJnYXVzc2lhbi1maW5hbC5ucHoiOiB7DQogICAgICAic2hhMjU2IjogIjg4ZmNmYTNkNDFlY2FmODE3OWE2MjhhZmI3NDJkZjdkY2Y5YmJmOTg4NTA5ZGE1ZjUyYzE4Yzc3NGJkMTQyYWIiLA0KICAgICAgImJ5dGVzIjogMTIyNjk2NzANCiAgICB9LA0KICAgICJzeW50aGV0aWMtYW5jaG9ycy5ucHoiOiB7DQogICAgICAic2hhMjU2IjogImI3MjZkN2YwZDlmNGU2ZjM5M2U1OGUwZDlkNDZmOWE2MDk4NDM0OTAxNTQ3ZDBhOGU1ZDhiMTA3OGFkNmI0NGUiLA0KICAgICAgImJ5dGVzIjogMTM3NTI3DQogICAgfQ0KICB9LA0KICAic3RhdHVzIjogIkNPTVBMRVRFX0NPTVBBQ1RfRklOQUxfSU5GRVJFTkNFX09OTFkiLA0KICAicHVibGljX3VuaXF1ZV9yb3dzIjogMjgzOTAsDQogICJwdWJsaWNfdXBkYXRlcyI6IDE2OTYNCn0NCg=="}}', 'diagnostic600_runner.py': '"""Actual mounted-input audit; diagnostic state does not represent .600 inference."""\nimport ast,base64,glob,json,os,shutil\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom numeric_transport import save_numeric\nHERE=Path(__file__).resolve().parent\nsource=(HERE/\'baseline600.py\').read_text();tree=ast.parse(source)\nnamespace={\'glob\':glob,\'json\':json,\'os\':os,\'INPUT_DIR\':\'/kaggle/input\'}\nfor node in tree.body:\n    if isinstance(node,ast.FunctionDef) and node.name in [\'find\',\'find_base\']:\n        exec(compile(ast.Module(body=[node],type_ignores=[]),\'<original-input-discovery>\',\'exec\'),namespace)\nrequired=[\'pool_v1.parquet\',\'lora8b_v1/meta.json\',\'lora14b_v1/meta.json\']\ninventory={pattern:namespace[\'find\'](pattern) for pattern in required}\nfor model in [{\'model_type\':\'qwen3\',\'hidden_size\':4096,\'num_hidden_layers\':36},{\'model_type\':\'qwen3\',\'hidden_size\':5120,\'num_hidden_layers\':40}]:\n    path=Path(namespace[\'find_base\'](model));assert (path/\'model.safetensors.index.json\').is_file();inventory[str(model)]=str(path)\n# Inventory the old feature-stack\'s literal file patterns without initializing its models.\nold=next(ast.literal_eval(n.value) for n in tree.body if isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id==\'_GAUSSIAN_SOURCE\' for t in n.targets))\nold_namespace={}\nexec(compile(old.split(\'# ---- data\')[0],\'<original-old-stack-specifications>\',\'exec\'),old_namespace)\nold_tree=ast.parse(old)\nfunction=next(n for n in old_tree.body if isinstance(n,ast.FunctionDef) and n.name==\'find_checkpoint\')\nexec(compile(ast.Module(body=[function],type_ignores=[]),\'<old-checkpoint-discovery>\',\'exec\'),old_namespace)\nfor _,model_type,hidden,*_ in old_namespace[\'LM_HEADS\']:\n    path=old_namespace[\'find_checkpoint\'](model_type,hidden);assert path,(model_type,hidden);inventory[model_type+str(hidden)]=path\nfor _,name,*_ in old_namespace[\'ENC_HEADS\']:\n    path=old_namespace[\'find_checkpoint\'](None,dir_name=name);assert path,name;inventory[name]=path\nfor _,_,runs in old_namespace[\'LORA_RUNS\']:\n    for name,pattern,norms,anchor in runs:\n        hits=sorted(glob.glob(\'/kaggle/input/\'+pattern,recursive=True));assert hits,name\n        ck=Path(hits[0]).parent\n        assert (ck/norms).is_file() and (ck/\'readouts.npz\').is_file() and (ck/\'adapter/adapter_model.safetensors\').is_file(),name\n        inventory[name]=str(ck)\nfor _,head,_ in old_namespace[\'TOKATTN_HEADS\']:\n    hits=list(Path(\'/kaggle/input\').rglob(head));assert hits,head;inventory[head]=str(hits[0])\nfor node in ast.walk(ast.parse(old)):\n    if isinstance(node,ast.Call) and isinstance(node.func,ast.Name) and node.func.id==\'find\' and node.args and isinstance(node.args[0],ast.Constant) and isinstance(node.args[0].value,str):\n        pattern=node.args[0].value\n        if pattern not in inventory:inventory[pattern]=namespace[\'find\'](pattern)\ntests=[p for p in [Path(\'/kaggle/input/competitions/name-that-llm/test.csv\'),Path(\'/kaggle/input/name-that-llm/test.csv\')] if p.is_file()];assert len(tests)==1\nquery=pd.read_csv(tests[0],dtype=str);ids=query.id.to_numpy(dtype=str)\nconfig=json.loads((HERE/\'volume-spec.json\').read_text());assert config[\'manifest_sha256\']\nlabels=json.loads((HERE/\'olmo-spec.json\').read_text())[\'classes\']\ncache=Path(os.environ[\'NTL_CANDIDATE_TEMP\'])\nasset=Path(os.environ[\'NTL_DIAGNOSTIC_VOLUME_ASSETS\']);asset.mkdir()\nbundle=json.loads((HERE/\'diagnostic-assets.json\').read_text())\nfor rel,encoded in bundle[\'binary_literal_files\'].items():\n    dest=asset/rel;dest.parent.mkdir(exist_ok=True,parents=True);dest.write_bytes(base64.b64decode(encoded))\nfor rel,suffix in bundle[\'copied_files\'].items():\n    hits=[p for p in Path(\'/kaggle/input\').rglob(Path(suffix).name) if str(p).endswith(suffix)]\n    assert len(hits)==1,(suffix,hits)\n    dest=asset/rel;dest.parent.mkdir(exist_ok=True,parents=True);shutil.copyfile(hits[0],dest)\nsave_numeric(cache/\'baseline-state.npz\',ids=ids,labels=np.asarray(labels,dtype=str),raw=np.zeros((len(ids),18),dtype=np.float64))\nsave_numeric(cache/\'baseline-groups.npz\',ids=ids,groups=np.arange(len(ids),dtype=np.int64),eligible=np.zeros(len(ids),dtype=bool))\nPath(\'/kaggle/working/mount-preflight.json\').write_text(json.dumps({\'status\':\'PASS_ACTUAL_MOUNTS_AND_LITERAL_SOURCE_PATTERNS\',\'inventory\':inventory,\'baseline_is_synthetic_stub\':True,\'full_0600_runtime_tested\':False},indent=2))\n', 'candidate-config.json': '{\n  "source_ready": true,\n  "OLMo_plain_raw_coefficient": 0.03,\n  "Smol_raw_coefficient": 0.03,\n  "new_Smol_Gaussian_weight": 0.0,\n  "source_sha256": {\n    "baseline600_runner.py": "a7980035b17d444ef0c59cb850c4cf2bc7ac79c41f42fc79df2ad10140b5be0b",\n    "volume_smol_worker.py": "15da84316056febded61bdd6e3d9ac6bbcc3a29592c5acf2da4581a11a40e90b",\n    "run600.py": "583cb14c239d334d9135a63769c0d76375406e5562a02b762e4f6aec08f0640f",\n    "finish600.py": "c5237fafffe38a7a767e49d74134011cc496f9bfdc5a40cf4fb0360d6506722a",\n    "ablate_legacy_qwen.py": "cae77c63fcee0ecaa99b136eb6df6d6bc6fdbcab66dec61d7642b559bf06e25a",\n    "baseline600.py": "7ab8127fe82378dd6aad9d08c440b3bc491e1dc8eec9367bfed67a47748dcd56",\n    "numeric_transport.py": "0c06ff559a0320e6e52cb4620274db10ff4a599899145d87b459c76ecdaa1b81",\n    "compute_olmo.py": "f0cbad0f32ad17c6b3c723addbcf64027651ac414c7e81775b596ca56eca8198",\n    "olmo_component.py": "32c7f5c4f89bced4e21bc5e852e1f9086b95d10842f009d5799a5c10be875f45",\n    "olmo_owned_path.py": "8e702264a084b93be8dbce9b1c50e59cd9b12ca0ded11ff67a7da73bbc642649",\n    "olmo-spec.json": "0f7bc46bb9be6879a53baa829e97fc935e92493f349380fb4b1a9d7c1b8ddd48",\n    "worker_barrier.py": "c01a5271e0b9bc296810d47309cd12a2791d4741839d15534b677d978119158e",\n    "smol_owned_path.py": "20dabafd3f423196863ce3d1fbd4caa8b196bd517a88422561615db003d1e620",\n    "smol_readout.py": "add20d61c93c891ea828aedf3cbf9509cedca9618ba3eee85cff9386f8e60ad0",\n    "center_readout.py": "49796a3787b2b342d71118624a45bb5932b0c66fee95322dcf4c2399cb125c30",\n    "gaussian_core.py": "4b5a1f0c17df93e31ae424ae176135c2292bdd20f70088fd4a1eee76335f3020",\n    "volume-spec.json": "b736db7cc5187ff9d155cb7773be6fb4d9c46548be189af5a2a72b56663fe88f",\n    "diagnostic-assets.json": "34738397392f5482f94353addb303b521a9b7d4bc3376675ee6d8cfe7efd7fde",\n    "diagnostic600_runner.py": "52861712e7b1abe3752bf88ca3d123a5c926819b391f274c00be772e7a884546"\n  },\n  "baseline_ref": "kirilltym/0-600-teach-the-model-remove-the-prompt",\n  "baseline_version": 20,\n  "public_unique_rows": 28390,\n  "public_updates": 1696,\n  "full_0600_OOF_available": false,\n  "source_contribution": "Independently trained full-external Smol plus native OLMo; preserved attributed .600 pre-balancing method",\n  "exclude_owned_official_qwen": true\n}'}
for name,source in FILES.items():
 (p/name).write_text(source)
 if name.endswith('.py'):ast.parse(source,filename=name)


## Predict and balance

In [ ]:
subprocess.run([sys.executable,'-u',str(p/'run600.py')],check=True,timeout=10900)